# AHR Experiment 2 — Final deployable agentic system

**12 controlled synthetic cases × 5 repeats = 60 formal model executions · nurse comparator is post-run only · paid calls OFF by default**

This notebook evaluates the final AHR architecture developed after Experiment 1 and its parallel evidence/technical follow-ups.

### Frozen architecture
- GPT-6 Luna agentic workflow with bounded tools.
- State-aware sound evidence validation: `NO_CHANGE` is evaluated against the **resulting retained sound state**.
- Curated model-visible evidence: the tested rich 12-source core plus six pre-existing Experiment-2-relevant context/action sources; E36 is supplied separately as a non-citable lux/dBA measurement convention.
- One **compact no-tools correction** if the first final answer is invalid.
- If that correction still fails for a purely mechanical JSON/schema/calculation reason, one **mechanical repair** is permitted.
- First-pass validity, repair use and final validity remain separately recorded.
- Adds `clinical_review.recommended` (`YES`/`NO`) for comparison with the nurse's independent clinical-assessment flag. This is advisory and non-diagnostic.

### Comparator separation
The frozen NURSE-001 CSV is **not read, hashed, imported or referenced by any inference/prompt cell**. It is loaded only in the final post-run analysis cell after the 60 model records already exist.

The notebook contains no expected case answers and does not score a model choice as correct during inference merely because a nurse later made the same choice.


### v4 infrastructure-only freeze fix
This version is model-behavior-equivalent to v3. The only change is the implementation fingerprint used by the frozen run contract: nested Python code objects are now serialized deterministically instead of using `repr(code.co_consts)`, which can contain runtime memory addresses and falsely change across Colab/kernel sessions. No prompt, case facts, evidence, schema, tools, validator behavior, clinical-review definition, nurse data, or model settings were changed.


In [ ]:
# 1 · Small Python libraries (CPU is sufficient)
%pip -q install "requests>=2.32,<3" "pandas>=2.2,<3" "jsonschema>=4.23,<5"


In [ ]:
# 2 · Controls: change only RUN_STAGE / APPROVE_PAID_CALLS between stages
RUN_STAGE = 'offline'                    # 'offline' -> 'smoke' -> 'full'
APPROVE_PAID_CALLS = False              # explicit permission for any paid stage
USE_GOOGLE_DRIVE = True
DOWNLOAD_ZIP = True

MODEL = 'openai/gpt-6-luna'
PROVIDER = 'OpenAI'
REASONING_EFFORT = 'high'
SEND_TEMPERATURE = False
TEMPERATURE = 1.0
MAX_OUTPUT_TOKENS = 16384
AGENT_MAX_CALLS = 6                     # initial tool-capable workflow only
AGENT_MAX_TOOLS = 8
TRANSIENT_RETRIES = 1
REQUEST_TIMEOUT = 120
PACE_SECONDS = 5
REPEATS = 5
RUN_BUDGET_USD = 3.00
GLOBAL_BUDGET_USD = 3.00
MAX_REQUEST_RESERVATION_USD = 0.08
MAX_INPUT_PRICE_PER_M = 0.15
MAX_OUTPUT_PRICE_PER_M = 0.75
MAX_ESTIMATED_INPUT_TOKENS = 100000
MAX_HTTP_REQUESTS = 600
RUN_LABEL = 'AHR_Experiment2_FinalAgent_20261026_v4'
RESULTS_ZIP_NAME = 'AHR_EXPERIMENT2_FINAL_AGENT_V4_RESULTS.zip'
PROFILES = {'luna':{'model':MODEL,'provider_name':PROVIDER}}
MODEL_CHOICE = 'luna'
RUN_LIVE = RUN_STAGE!='offline'
CASE_IDS = [f'E2_{i:03d}' for i in range(1,13)]
assert RUN_STAGE in ('offline','smoke','full')
assert isinstance(APPROVE_PAID_CALLS,bool) and isinstance(USE_GOOGLE_DRIVE,bool)
print('STAGE',RUN_STAGE,'| planned full workflows',12*REPEATS,'| key needed',RUN_LIVE)


In [ ]:
# 3 · Storage, hashing and write-once run identity
import copy,csv,hashlib,json,math,os,re,time,uuid,zipfile,base64,zlib,types,inspect
from datetime import datetime,timezone
from pathlib import Path
from contextlib import contextmanager
import requests,pandas as pd
from jsonschema import Draft202012Validator
from IPython.display import display

def stamp():return datetime.now(timezone.utc).isoformat()
def encode(x):return json.dumps(x,sort_keys=True,ensure_ascii=False,allow_nan=False)
def digest(x):return hashlib.sha256(encode(x).encode()).hexdigest()
def require(condition,msg):
    if not condition:raise ValueError(msg)
def save_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False));tmp.replace(path)
def write_frozen(path,obj):
    path=Path(path)
    if path.exists(): require(encode(json.loads(path.read_text()))==encode(obj), 'Frozen file changed: '+str(path)+'. Choose a new RUN_LABEL.')
    else:save_json(path,obj)

require(not RUN_LIVE or APPROVE_PAID_CALLS, 'Live work requires APPROVE_PAID_CALLS=True.')
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/AHR_Luna_30_Study')
else:
    BASE=Path('/content/AHR_Luna_30_Study') if Path('/content').exists() else Path.cwd()/'AHR_Luna_30_Study'
MASTER_ROOT=BASE/RUN_LABEL;OUT=MASTER_ROOT/'luna';OUT.mkdir(parents=True,exist_ok=True)
require(re.fullmatch(r'[A-Za-z0-9_-]+',RUN_LABEL) is not None,'Invalid RUN_LABEL')

@contextmanager
def generation_lock(directory):
    lock=Path(directory)/'generation.lock'
    try:fd=os.open(str(lock),os.O_CREAT|os.O_EXCL|os.O_WRONLY,0o600)
    except FileExistsError:raise RuntimeError('Interrupted / concurrent run lock; examine saved logs before resuming.')
    try:
        os.write(fd,stamp().encode());os.close(fd)
        yield
    finally:lock.unlink(missing_ok=True)

for key in ('MAX_OUTPUT_TOKENS','AGENT_MAX_CALLS','AGENT_MAX_TOOLS','MAX_HTTP_REQUESTS','MAX_ESTIMATED_INPUT_TOKENS','REPEATS'):
    require(type(globals()[key]) is int and globals()[key]>0,key+' must be positive integer')
for key in ('RUN_BUDGET_USD','MAX_REQUEST_RESERVATION_USD','MAX_INPUT_PRICE_PER_M','MAX_OUTPUT_PRICE_PER_M','REQUEST_TIMEOUT'):
    require(type(globals()[key]) in (int,float) and globals()[key]>0 and math.isfinite(globals()[key]),key+' must be finite positive')
require(0<=PACE_SECONDS<=120 and TRANSIENT_RETRIES in (0,1),'Invalid pacing/retry')
print('Results folder:',MASTER_ROOT)


In [ ]:
# 4 · Frozen Experiment 2 evidence representation
# The 18 research entries were selected from the already-prepared Experiment 2 corpus based on case domains,
# not from nurse response fields. E36 is a separate measurement convention and cannot justify an action.
EVIDENCE = [{'id': 'E01', 'kind': 'research', 'topic': 'lighting', 'source': 'Giménez et al. (2017)', 'proposal_key': 'gimenez2017lighting', 'doi': '10.1111/jsr.12470', 'url': 'https://doi.org/10.1111/jsr.12470', 'verified_from': 'Publisher summary, methods and selected results sections', 'checked_date': '2026-09-28', 'study_design': 'Controlled clinical study; 196 cardiology patients', 'exposure': 'Combined 24-hour lighting: brighter daytime exposure and lower night lighting over hospitalization.', 'finding': 'Actigraphic sleep duration increased by about 5.9 minutes per hospitalization day; lighting appraisal improved. Subjective sleep, alertness, anxiety and depression did not differ between groups.', 'limitations': 'Combined exposure over days does not isolate tone, establish acute calming, or validate device percentages. One participant withdrew with intervention-induced migraine.', 'role': 'action_support', 'context_fields': ['lighting'], 'lighting_actions': ['INCREASE', 'DECREASE', 'NO_CHANGE'], 'sounds': [], 'condition': 'time_of_day', 'study_context': {'population_setting': '196 cardiology-ward patients; mean age 66.5 ± 13.1 years; median hospitalization 5 days in both study arms.', 'design_sample': 'Controlled clinical trial comparing standard patient-room lighting with an interventional lighting system.', 'intervention_exposure': 'Dynamic 24-hour light/dark cycle with low nocturnal light exposure and 2 hours of bright daytime light at 1750 lux.', 'comparator': 'Standardly lit patient rooms.', 'timing_duration': 'Lighting exposure continued across hospitalization; outcomes were assessed over the admission.', 'outcomes_measured': 'Wrist actigraphy plus questionnaires covering sleep quality, alertness, anxiety, depression and lighting appraisal.', 'results_detail': 'Subjective sleep, alertness, anxiety and depression did not differ between groups. Lighting appraisal was better in intervention rooms for patients (P<0.001) and staff (P<0.005). Actigraphic sleep duration increased by 5.9 min per hospitalization day (95% CI 0.6–11.2; P=0.03); after 5 days the difference was 29 min (7.3%).', 'study_limitations_context': 'The intervention changed the full 24-hour lighting pattern rather than a single isolated lighting component. One participant withdrew because of an intervention-induced migraine.'}}, {'id': 'E02', 'kind': 'research', 'topic': 'sound', 'source': 'Fan & Baharum (2024)', 'proposal_key': 'fan2024natural', 'doi': '10.1080/10253890.2024.2402519', 'url': 'https://doi.org/10.1080/10253890.2024.2402519', 'verified_from': 'Author abstract via PubMed/Europe PMC', 'checked_date': '2026-09-28', 'study_design': 'Systematic review and meta-analysis', 'exposure': 'Natural sounds compared with quiet.', 'finding': 'Differences favoured natural sounds for heart rate, blood pressure and respiratory rate; mean arterial pressure, perceived stress and oxygen saturation did not differ significantly.', 'limitations': 'Mixed outcomes; abstract alone does not establish effects for this individual, a safe dose or absence of adverse effects.', 'role': 'action_support', 'context_fields': ['sound'], 'lighting_actions': [], 'sounds': ['NATURE_SOUNDS'], 'condition': 'sound_permission', 'study_context': {'population_setting': 'Systematic review and meta-analysis of human studies comparing exposure to natural sounds with a quiet environment.', 'design_sample': 'Systematic review and meta-analysis; the published abstract reports pooled comparisons across several physiological and self-reported stress outcomes.', 'intervention_exposure': 'Natural-sound exposure, including sounds such as birdsong, compared with quiet.', 'comparator': 'Quiet environment.', 'timing_duration': 'Exposure timing and duration varied across the included studies.', 'outcomes_measured': 'Heart rate, blood pressure, respiratory rate, mean arterial pressure, perceived stress and oxygen saturation.', 'results_detail': 'Natural sounds differed significantly from quiet for heart rate (P=0.006), blood pressure (P=0.001) and respiratory rate (P=0.032). No significant differences were reported for mean arterial pressure (P=0.407), perceived stress, or SpO2 (P=0.251).', 'study_limitations_context': 'Effects were not consistent across all measured outcomes, and exposure protocols varied across the included literature.'}}, {'id': 'E03', 'kind': 'research', 'topic': 'physiology', 'source': 'Shaffer & Ginsberg (2017)', 'proposal_key': 'shaffer2017overview', 'doi': '10.3389/fpubh.2017.00258', 'url': 'https://doi.org/10.3389/fpubh.2017.00258', 'verified_from': 'Open-access article; selected metric and interpretation sections', 'checked_date': '2026-09-28', 'study_design': 'Review of HRV metrics and norms', 'exposure': 'Measurement duration, physiological influences and interpretation.', 'finding': 'HRV interpretation depends on recording duration and context, including respiration.', 'limitations': 'An RMSSD change alone does not establish psychological state or a patient-specific uncertainty interval.', 'role': 'context_only', 'context_fields': ['physiology'], 'lighting_actions': [], 'sounds': [], 'condition': 'always', 'study_context': {'population_setting': 'Methodological review covering HRV measurement and norms across healthy and clinical contexts rather than a single patient cohort.', 'design_sample': 'Narrative/methodological review of commonly used time-domain, frequency-domain and nonlinear HRV metrics and their interpretation.', 'intervention_exposure': 'No intervention; the review discusses how HRV values depend on recording duration, physiological mechanisms and measurement context.', 'comparator': 'Not an intervention-comparator study.', 'timing_duration': 'The review distinguishes long-term (~24 h), short-term (~5 min) and ultra-short-term (<5 min) HRV recordings.', 'outcomes_measured': 'HRV metrics including RMSSD, SDNN and frequency-domain measures, with discussion of normative interpretation and recording requirements.', 'results_detail': 'RMSSD is a time-domain measure based on successive normal-to-normal interval differences and is commonly used for short-term HRV. The review emphasizes that HRV interpretation depends on recording duration and physiological context, including respiration.', 'study_limitations_context': 'This is a measurement/interpretation review rather than an intervention trial, and HRV values are sensitive to methodological and physiological context.'}}, {'id': 'E04', 'kind': 'prototype_rule', 'topic': 'lighting', 'source': 'AHR optional rationale: retain brightness', 'url': None, 'finding': 'Retaining the current brightness is an available choice when compatible with the case. This entry supplies a non-research rationale for that choice. Tone is selected separately within the case restrictions.', 'limitations': 'This entry imposes no lighting lock, default preference, veto, or requirement to retain brightness. Other choices may use another applicable basis. A chosen increase/decrease needs its own applicable support; do not cite this retention rationale for those actions. No physiological or sleep benefit is established.', 'lighting_actions': ['NO_CHANGE'], 'sounds': [], 'condition': 'always', 'role': 'optional_action_basis', 'context_fields': ['lighting'], 'decision_force': 'OPTIONAL_ACTION_BASIS', 'mandatory': False, 'restricts_other_actions': False}, {'id': 'E05', 'kind': 'prototype_rule', 'topic': 'sound', 'source': 'AHR optional rationale: preference-compatible sound', 'url': None, 'finding': 'Silence, retaining permissible existing sound, or offering any explicitly preferred allowed sound may respect patient choice. All active sounds remain subject to staff permission and patient prohibitions.', 'limitations': 'Preference compatibility is not efficacy evidence. A stated preference does not require playback; explain a decision to retain silence when a permitted preference is supplied.', 'lighting_actions': [], 'sounds': ['NONE_SILENCE', 'NO_CHANGE', 'PREFERRED_CALMING_MUSIC', 'NATURE_SOUNDS', 'WHITE_NOISE'], 'condition': 'preference_compatible', 'role': 'optional_action_basis', 'context_fields': ['sound'], 'decision_force': 'OPTIONAL_ACTION_BASIS', 'mandatory': False, 'restricts_other_actions': False}, {'id': 'E06', 'kind': 'prototype_rule', 'topic': 'physiology', 'source': 'AHR development convention', 'url': None, 'finding': 'Report signed current-minus-reference changes; HR/RMSSD percentages use the reference denominator. Report EDA absolute change only. Missing or poor-quality pairs are unavailable.', 'limitations': 'These calculations are descriptive. EDA percentage reporting is disabled by design, not because absolute comparisons are invalid.', 'lighting_actions': [], 'sounds': [], 'condition': 'always', 'role': 'calculation_convention', 'context_fields': ['physiology'], 'decision_force': 'CALCULATION_CONVENTION'}, {'id': 'E07', 'kind': 'research', 'topic': 'sound', 'source': 'Papathanassoglou et al. (2025)', 'proposal_key': 'papathanassoglou2025sound', 'doi': '10.1016/j.aucc.2024.101148', 'url': 'https://doi.org/10.1016/j.aucc.2024.101148', 'verified_from': 'Author abstract via PubMed/Europe PMC', 'checked_date': '2026-09-28', 'study_design': 'Systematic review: 59 articles, 37 music and 22 nonmusic', 'exposure': 'Randomized studies of music and nonmusic sound; only one direct comparison.', 'finding': 'Reported benefits across pain, sleep and stress/anxiety outcomes; a larger music literature does not establish superiority over nonmusic sound.', 'limitations': 'Heterogeneous interventions and outcomes; no fixed AHR playback dose or individual benefit follows from the abstract.', 'role': 'action_support', 'context_fields': ['sound'], 'lighting_actions': [], 'sounds': ['PREFERRED_CALMING_MUSIC'], 'condition': 'sound_permission', 'study_context': {'population_setting': 'Systematic review of randomized studies involving general ICU populations, including patients with and without the ability to communicate.', 'design_sample': '59 included articles: 37 music interventions and 22 nonmusic sound interventions; only one study directly compared music with nonmusic sound.', 'intervention_exposure': 'Music and nonmusic sound interventions delivered in ICU settings; protocols varied across studies.', 'comparator': 'Standard care and/or noise-reduction controls depending on the included trial.', 'timing_duration': 'Intervention timing and duration varied across the randomized studies.', 'outcomes_measured': 'Pain, sleep, stress/anxiety, relaxation, cortisol, sedative/analgesic use, vital signs and haemodynamic measures.', 'results_detail': 'Both slow-tempo music and nonmusic sound interventions were associated in included trials with improvements in several outcomes including pain, sleep, stress/anxiety and relaxation. The review reported a larger evidence base for music than nonmusic sound for stress biomarkers, vital signs and haemodynamic measures.', 'study_limitations_context': 'The evidence base was much larger for music than nonmusic sound, and only one included study directly compared the two intervention categories.'}}, {'id': 'E08', 'kind': 'research', 'topic': 'sound', 'source': 'Ding et al. (2025)', 'proposal_key': 'ding2025whitenoise', 'doi': '10.1016/j.sleep.2025.106869', 'url': 'https://doi.org/10.1016/j.sleep.2025.106869', 'verified_from': 'Author abstract via PubMed/Europe PMC', 'checked_date': '2026-09-28', 'study_design': 'Meta-analysis: 12 randomized trials, 1,301 participants across age groups', 'exposure': 'White noise for sleep; adult, older-adult and infant populations.', 'finding': 'Adult and older-adult pooled sleep-quality scores improved; adult results had substantial heterogeneity.', 'limitations': 'Infant findings cannot simply transfer to adult hospital cases. Sleep outcomes do not establish acute stress relief or an exact device volume.', 'role': 'action_support', 'context_fields': ['sound'], 'lighting_actions': [], 'sounds': ['WHITE_NOISE'], 'condition': 'sleep_goal', 'study_context': {'population_setting': '12 randomized controlled trials with 1,301 participants: 356 infants/young children, 648 adults aged 18–64, and 297 adults aged ≥65. The review included critically ill and non-critically ill populations.', 'design_sample': 'Systematic review and meta-analysis of randomized controlled trials of white noise and sleep.', 'intervention_exposure': 'White-noise interventions; protocols and settings varied across trials.', 'comparator': 'Control conditions from the included RCTs, such as usual care or no white-noise intervention.', 'timing_duration': 'Exposure duration varied across studies; analyses were stratified by age group and clinical setting.', 'outcomes_measured': 'Pittsburgh Sleep Quality Index (PSQI), total sleep time, sleep efficiency, number of awakenings and wake after sleep onset.', 'results_detail': 'In adults, pooled PSQI decreased by 3.70 points (95% CI -4.90 to -2.50; P<0.001); in older adults by 2.71 points (95% CI -4.98 to -0.44; P=0.02). PSQI also decreased in ICU patients (MD -4.04, 95% CI -6.35 to -1.73; P<0.001) and non-ICU settings (MD -2.61, 95% CI -3.84 to -1.38; P<0.001). Infant/young-child outcomes were mixed across total sleep time, sleep efficiency and awakenings.', 'study_limitations_context': 'Adult PSQI analyses showed substantial heterogeneity. Several subgroup outcomes contained few studies, limiting sensitivity analyses.'}}, {'id': 'E09', 'kind': 'research', 'topic': 'sound', 'source': 'Nien et al. (2026)', 'proposal_key': 'nien2026whitenoise', 'doi': '10.1016/j.iccn.2025.104239', 'url': 'https://doi.org/10.1016/j.iccn.2025.104239', 'verified_from': 'Publisher abstract and study methods (corrected playback time)', 'checked_date': '2026-10-15', 'study_design': 'Randomized trial: 58 ICU patients in Taiwan', 'exposure': '40–50 dB white noise from 23:00 to 06:00 for three consecutive nights versus routine care.', 'finding': 'Some self-reported sleep measures improved during follow-up.', 'limitations': 'Small single-setting trial; overnight sound is not equivalent to a 15-minute simulation. Volume percentage has no established conversion to decibels.', 'role': 'action_support', 'context_fields': ['sound'], 'lighting_actions': [], 'sounds': ['WHITE_NOISE'], 'condition': 'sleep_goal', 'study_context': {'population_setting': '58 ICU patients in a medical center in southern Taiwan; 29 intervention and 29 control; mean age 61.7 ± 13.6 years; 56.9% male.', 'design_sample': 'Experimental longitudinal randomized controlled trial using block randomization, conducted March 2023 to August 2024.', 'intervention_exposure': 'White noise at 40–50 dB from 23:00 to 06:00 for three consecutive nights.', 'comparator': 'Routine ICU care without white noise.', 'timing_duration': 'Three consecutive overnight exposures; sleep was measured at baseline and on days 1, 3 and 5.', 'outcomes_measured': 'Verran and Snyder-Halpern Sleep Scale (VSHSS), including total sleep score, sleep disturbance, sleep effectiveness and sleep supplementation.', 'results_detail': 'Compared with control, the intervention group showed greater reductions in total sleep scores and sleep disturbances on days 1, 3 and 5 (P<0.01), improved sleep effectiveness on days 3 and 5 (P<0.01), and higher sleep supplementation on day 3 (P<0.01).', 'study_limitations_context': 'Single-center study with a small sample; outcomes were subjectively assessed using VSHSS.'}}, {'id': 'E10', 'kind': 'research', 'topic': 'sound', 'source': 'Saskovets et al. (2025)', 'proposal_key': 'saskovets2025sound', 'doi': '10.2196/69120', 'url': 'https://doi.org/10.2196/69120', 'verified_from': 'Author abstract via PubMed/Europe PMC', 'checked_date': '2026-09-28', 'study_design': 'Review: 34 clinical and laboratory studies', 'exposure': 'Music and sound interventions across settings.', 'finding': 'Physiological and self-reported responses vary with preferences, delivery and cultural context; some adverse responses were reported.', 'limitations': 'Heterogeneous findings do not establish a universally beneficial or harmless sound.', 'role': 'context_only', 'context_fields': ['sound'], 'lighting_actions': [], 'sounds': [], 'condition': 'always', 'study_context': {'population_setting': 'Scoping review of adult clinical and laboratory research on sound interventions and mental stress.', 'design_sample': '34 studies published from 1990–2024, including randomized controlled trials, clinical trials and laboratory experiments.', 'intervention_exposure': 'Music, natural sounds and speech, with variation in sound type, delivery method and listening context.', 'comparator': 'Comparators varied across studies and included control or alternative sound conditions.', 'timing_duration': 'Intervention timing and duration varied across the included studies.', 'outcomes_measured': 'Physiological markers such as HRV, cortisol and blood pressure, alongside self-reported stress and related outcomes.', 'results_detail': 'Music—especially classical and self-selected music—was frequently associated with reductions in physiological stress markers. Nonmusic sounds also showed potential effects, while responses varied with individual preferences and contextual factors; some included studies reported adverse responses.', 'study_limitations_context': 'The included studies were heterogeneous in setting, sound type, participant characteristics, delivery protocol and outcome measures.'}}, {'id': 'E11', 'kind': 'research', 'topic': 'lighting', 'source': 'Martins et al. (2025)', 'proposal_key': 'martins2025light', 'doi': '10.1016/j.neubiorev.2025.106241', 'url': 'https://doi.org/10.1016/j.neubiorev.2025.106241', 'verified_from': 'Author abstract via PubMed/Europe PMC', 'checked_date': '2026-09-28', 'study_design': 'Systematic review: 25 studies, all assessed at high risk of bias', 'exposure': 'Light exposure and vagally mediated HRV in healthy and clinical populations.', 'finding': 'Higher-illuminance blue light generally corresponded to lower vmHRV; lower-illuminance warm light showed potential increases, with inconsistent findings.', 'limitations': 'High risk of bias and heterogeneity prevent treating warm tone as proven calming or diagnosing state from HRV.', 'role': 'context_only', 'context_fields': ['lighting'], 'lighting_actions': [], 'sounds': [], 'condition': 'always', 'study_context': {'population_setting': 'Systematic review of healthy and clinical populations exposed to visible light of varying illuminance and color.', 'design_sample': '25 studies included after screening 24,673 records; all included studies were rated high risk of bias.', 'intervention_exposure': 'Ocular exposure to visible light varying in illuminance and color, including blue, red/warm and white-light conditions.', 'comparator': 'Control conditions differed across studies.', 'timing_duration': 'Exposure protocols and durations varied substantially across the literature.', 'outcomes_measured': 'Vagally mediated heart-rate variability (vmHRV) under different light-exposure conditions.', 'results_detail': 'Higher-illuminance blue light was generally associated with reduced vmHRV, while lower-illuminance warm-colored light such as red or dim white showed potential increases in vmHRV. Findings were inconsistent across studies.', 'study_limitations_context': 'All included studies were judged high risk of bias, with substantial heterogeneity in populations, exposure protocols and control conditions.'}}, {'id': 'E12', 'kind': 'research', 'topic': 'lighting', 'source': 'Linders et al. (2026)', 'proposal_key': 'linders2026lighting', 'doi': '10.1016/j.iccn.2026.104421', 'url': 'https://doi.org/10.1016/j.iccn.2026.104421', 'verified_from': 'Author abstract via PubMed/Europe PMC', 'checked_date': '2026-09-28', 'study_design': 'Systematic review: 28 ICU studies, 14 intervention studies', 'exposure': 'Naturalistic and intervention lighting research in intensive care.', 'finding': 'Marked heterogeneity prevented definitive outcome conclusions and clear lighting-design guidance.', 'limitations': 'ICU evidence cannot establish universal parameters or a guaranteed sleep benefit.', 'role': 'context_only', 'context_fields': ['lighting'], 'lighting_actions': [], 'sounds': [], 'condition': 'always', 'study_context': {'population_setting': 'Systematic review of ICU patient-lighting research published between 2015 and 2026.', 'design_sample': '12,674 records screened; 28 studies included. Fourteen evaluated lighting interventions and fourteen examined naturalistic ICU lighting conditions.', 'intervention_exposure': 'Interventions included dynamic lighting, static lighting, bright-light therapy and multicomponent lighting; observational studies assessed existing ICU lighting conditions.', 'comparator': 'Comparators varied by study, including standard lighting, alternative lighting conditions, or observational comparisons.', 'timing_duration': 'Lighting protocols and observation periods varied substantially across studies.', 'outcomes_measured': 'A range of patient health outcomes and lighting-condition measures across ICU studies.', 'results_detail': 'Some studies reported potential benefits of specific lighting interventions, but findings were inconsistent and no definitive pattern emerged across the heterogeneous outcome and lighting measures.', 'study_limitations_context': 'Marked heterogeneity in interventions, lighting measurement, health outcomes and study methods; reporting of lighting conditions was inconsistent.'}}, {'id': 'E13', 'kind': 'research', 'topic': 'physiology', 'source': 'Posada-Quintero & Chon (2020)', 'proposal_key': 'posada2020innovations', 'doi': '10.3390/s20020479', 'url': 'https://doi.org/10.3390/s20020479', 'verified_from': 'Author abstract via PubMed/Europe PMC', 'checked_date': '2026-09-28', 'study_design': 'Systematic review of electrodermal measurement and processing', 'exposure': 'Tonic and phasic electrodermal activity and applications.', 'finding': 'EDA reflects sympathetic sweat-gland activity and is used across multiple arousal-related applications.', 'limitations': 'EDA alone is not a specific psychological stress diagnosis. The AHR absolute-only reporting rule is a prototype convention.', 'role': 'context_only', 'context_fields': ['physiology'], 'lighting_actions': [], 'sounds': [], 'condition': 'always', 'study_context': {'population_setting': 'Systematic review of electrodermal activity (EDA) measurement, signal processing and applications across psychophysiological and pathophysiological research.', 'design_sample': 'Methodological/systematic review rather than a single intervention cohort.', 'intervention_exposure': 'No single intervention; the review covers tonic and phasic EDA acquisition and processing methods.', 'comparator': 'Not an intervention-comparator study.', 'timing_duration': 'Measurement windows and processing approaches varied across the literature reviewed.', 'outcomes_measured': 'Tonic skin-conductance level, phasic skin-conductance responses, acquisition methods and signal-processing approaches.', 'results_detail': 'EDA reflects sympathetic innervation of sweat glands. The review describes use of EDA in emotional/cognitive stress research and applications including fatigue, pain, sleepiness, exercise recovery, epilepsy, neuropathies and depression.', 'study_limitations_context': 'EDA is used across diverse applications and processing methods; the review emphasizes measurement and processing methodology rather than a single disease- or intervention-specific effect.'}}, {'id': 'E14', 'kind': 'research', 'topic': 'lighting/sound', 'source': 'Mendonça et al. (2025)', 'proposal_key': 'mendonca2025nursing', 'doi': '10.1111/inr.13062', 'url': 'https://doi.org/10.1111/inr.13062', 'verified_from': 'Publisher abstract, eligibility and selected intervention-table sections', 'checked_date': '2026-09-28', 'study_design': 'Review: 13 studies, 1,975 hospitalized adults and older adults; noncritical wards', 'exposure': 'Nursing interventions including environmental, sensory, educational and individualized care.', 'finding': 'Reported sleep-focused approaches span multiple components, including light and noise management.', 'limitations': 'Critical-care patients were excluded; multicomponent evidence cannot establish an isolated light/sound effect or AHR dose.', 'role': 'context_only', 'context_fields': ['lighting', 'sound'], 'lighting_actions': [], 'sounds': [], 'condition': 'always', 'study_context': {'population_setting': 'Systematic review of hospitalized adults and older adults in noncritical hospital wards.', 'design_sample': '712 records identified; 13 studies included with 1,975 participants; findings synthesized narratively.', 'intervention_exposure': 'Sleep-promoting nursing interventions spanning educational/communication strategies, dietary and sensory interventions, symptom and environmental control, daily activity planning, sleep assessment/documentation and individualized nursing care.', 'comparator': 'Comparator conditions varied across the included studies.', 'timing_duration': 'Intervention schedules and follow-up varied across studies.', 'outcomes_measured': 'Sleep-related outcomes and implementation of nursing interventions intended to promote sleep.', 'results_detail': 'The review identified multiple recurring categories of nursing sleep interventions, including environmental and sensory approaches, and emphasized a multidimensional approach incorporating physical, psychosocial and relational aspects of care.', 'study_limitations_context': 'Critical-care patients were not the target population; the included interventions and outcome measures were heterogeneous.'}}, {'id': 'E15', 'kind': 'research', 'topic': 'lighting/sound', 'source': 'Wang et al. (2025)', 'proposal_key': 'wang2025nurse', 'doi': '10.1111/nicc.70028', 'url': 'https://doi.org/10.1111/nicc.70028', 'verified_from': 'Publisher abstract', 'checked_date': '2026-09-28', 'study_design': 'Before/after ICU quality-improvement project', 'exposure': 'Nurse-led SLEEP bundle including noise/light reduction, sensory aids and pharmacological/nonpharmacological measures.', 'finding': 'Reported sleep questionnaire scores rose from 62 to 70 after implementation.', 'limitations': 'Bundle and before/after design do not isolate lighting or sound causality. Medication components are outside AHR.', 'role': 'context_only', 'context_fields': ['lighting', 'sound'], 'lighting_actions': [], 'sounds': [], 'condition': 'always', 'study_context': {'population_setting': 'Quality-improvement study in a 12-bed surgical ICU in Wuhan, China.', 'design_sample': 'Quasi-experimental before/after evidence-based quality-improvement programme using the Framework of Evidence-based Continuous Quality Improvement and the Ottawa Model of Research Use.', 'intervention_exposure': 'Nurse-led SLEEP Bundle: Sleep initiative; Light control; Eye mask/earplugs; Environment noise cancellation; and Provision of non-pharmacological support (including aromatherapy/music) and pharmacological support (including dexmedetomidine/painkillers).', 'comparator': 'Pre-implementation ICU care versus post-implementation care.', 'timing_duration': 'Programme-level implementation with before/after assessment; the abstract does not isolate a single environmental exposure duration.', 'outcomes_measured': "Patient-perceived sleep quality using the Richards-Campbell Sleep Questionnaire, nurses' knowledge/attitudes/actions and adherence to sleep-promotion practices.", 'results_detail': 'Median Richards-Campbell Sleep Questionnaire score increased from 62 (IQR 48–72) to 70 (IQR 62–76), with P<0.001. Nurse knowledge/action measures and adherence to several sleep-management practices also improved after implementation.', 'study_limitations_context': 'Multicomponent before/after quality-improvement design; the programme combined environmental, sensory, educational, non-pharmacological and pharmacological components.'}}, {'id': 'E23', 'kind': 'research', 'topic': 'hospital noise', 'source': 'Busch-Vishniac et al. (2005)', 'proposal_key': 'buschvishniac2005hopkins_noise', 'doi': '10.1121/1.2118327', 'url': 'https://doi.org/10.1121/1.2118327', 'verified_from': 'PubMed / Journal of the Acoustical Society of America record', 'checked_date': '2026-10-23', 'study_design': 'Hospital-wide acoustic survey at Johns Hopkins Hospital', 'exposure': 'Equivalent continuous sound pressure levels measured by location, frequency and time of day across Johns Hopkins Hospital.', 'finding': 'Average equivalent sound levels were generally in the 50–60 dB(A) range, and many units showed little or no nighttime reduction in noise.', 'limitations': 'Observational environmental measurement study; it does not test patient outcomes or show that any AHR sound choice improves sleep or recovery.', 'role': 'context_only', 'context_fields': ['sound'], 'lighting_actions': [], 'sounds': [], 'condition': 'always'}, {'id': 'E27', 'kind': 'research', 'topic': 'physiology', 'source': 'Vos et al. (2023)', 'proposal_key': 'vos2023wearable_stress', 'doi': '10.1016/j.ijmedinf.2023.105026', 'url': 'https://doi.org/10.1016/j.ijmedinf.2023.105026', 'verified_from': 'Verified AHR literature workbook + publication record', 'checked_date': '2026-10-23', 'study_design': 'Systematic literature review of wearable-device stress monitoring', 'exposure': 'Wearable HR, HRV, EDA and related signals used in stress-monitoring research.', 'finding': 'HRV, heart rate and EDA are commonly used contextual biomarkers, but datasets and methods vary and generalization across people/settings remains difficult.', 'limitations': 'Many datasets are small or short and biomarker validity/generalization are limited; does not justify diagnosing stress from one signal or threshold.', 'role': 'context_only', 'context_fields': ['physiology'], 'lighting_actions': [], 'sounds': [], 'condition': 'always'}, {'id': 'E29', 'kind': 'research', 'topic': 'sound / patient preference', 'source': 'Heiderscheit et al. (2022)', 'proposal_key': 'heiderscheit2022preferred_music', 'doi': '10.1089/jicm.2021.0446', 'url': 'https://doi.org/10.1089/jicm.2021.0446', 'verified_from': 'PubMed/PMC publication record', 'checked_date': '2026-10-23', 'study_design': 'Secondary analysis of preferred music selected by mechanically ventilated ICU patients enrolled in a randomized trial; N=126', 'exposure': 'Patient-selected music used within a music-listening intervention.', 'finding': 'Patients selected a wide range of music, reinforcing that clinically used music should be individualized rather than assumed to be universally calming.', 'limitations': 'Preference analysis does not establish that music is superior to silence or another sound for a given patient, and the population was mechanically ventilated ICU patients.', 'role': 'context_only', 'context_fields': ['sound'], 'lighting_actions': [], 'sounds': [], 'condition': 'always'}, {'id': 'E33', 'kind': 'research', 'topic': 'sound / sleep', 'source': 'Henriques et al. (2026)', 'proposal_key': 'henriques2026sound_sleep', 'doi': '10.1016/j.isci.2026.117698', 'url': 'https://doi.org/10.1016/j.isci.2026.117698', 'verified_from': 'PubMed + iScience publication record', 'checked_date': '2026-10-23', 'study_design': 'Systematic review; 15 studies (2012–2026), N=2,024 hospitalized patients, mostly intensive care', 'exposure': 'Calming music, nature sounds and white noise delivered from brief sessions to continuous overnight exposure.', 'finding': 'Sound-based interventions were associated overall with improved sleep quality, but effect sizes varied and several studies found no significant benefit.', 'limitations': 'Substantial heterogeneity in sound type, setting, timing, duration and outcome measurement; does not establish a universal best sound or AHR volume/duration.', 'role': 'action_support', 'context_fields': ['sound'], 'lighting_actions': [], 'sounds': ['PREFERRED_CALMING_MUSIC', 'NATURE_SOUNDS', 'WHITE_NOISE'], 'condition': 'sleep_goal'}, {'id': 'E37', 'kind': 'research', 'topic': 'smart patient room / patient autonomy', 'source': 'Dawson, Kauffman & Wiese (2023)', 'proposal_key': 'dawson2023patient_smart_room', 'doi': '10.1145/3544548.3580757', 'url': 'https://doi.org/10.1145/3544548.3580757', 'verified_from': 'ACM/PeDEL publication record', 'checked_date': '2026-10-23', 'study_design': 'Peer-reviewed CHI qualitative study; semi-structured interviews with 20 rehabilitation-hospital patients', 'exposure': 'Smart patient rooms with digitally controllable entertainment and environment using touchscreen, voice and accessibility controls.', 'finding': 'Patients described smart-room technology as potentially transformative for people with reduced motor function by supporting independence and control over their surroundings; the study also identified implementation challenges and staff-support needs.', 'limitations': 'Single rehabilitation hospital and qualitative interviews; does not test automated physiological adaptation, clinical outcomes, or a specific light/sound intervention.', 'role': 'context_only', 'context_fields': ['lighting', 'sound'], 'lighting_actions': [], 'sounds': [], 'condition': 'always'}, {'id': 'E38', 'kind': 'research', 'topic': 'smart patient room / staff workflow', 'source': 'Dawson, Fisher & Wiese (2024)', 'proposal_key': 'dawson2024staff_smart_room', 'doi': '10.1145/3613904.3642201', 'url': 'https://doi.org/10.1145/3613904.3642201', 'verified_from': 'ACM/PeDEL publication record', 'checked_date': '2026-10-23', 'study_design': 'Peer-reviewed CHI qualitative study; 23 hospital employees across nine occupational groups', 'exposure': 'Real smart patient rooms with controllable lights, blinds, thermostat, TV and soundbar in a rehabilitation hospital.', 'finding': 'Employees reported benefits and drawbacks of smart-room controls; the work highlights that patients and staff can have different needs and suggests configurable limits such as light brightness or maximum audio volume to avoid overstimulation while preserving use.', 'limitations': 'Single rehabilitation hospital; qualitative study; does not validate AHR thresholds, automated recommendations, clinical efficacy or a particular environmental action.', 'role': 'context_only', 'context_fields': ['lighting', 'sound'], 'lighting_actions': [], 'sounds': [], 'condition': 'always'}]
ENVIRONMENT_MEASUREMENT_CONVENTION = {'id': 'E36', 'kind': 'measurement_convention', 'topic': 'environmental measurement', 'source': 'AHR environmental sensor convention', 'finding': 'Bedside illuminance (lux) and ambient sound level (dBA/LAeq) are 5-minute environmental measurements supplied as room context. They are not patient physiological measurements or clinical thresholds.', 'limitations': 'No validated direct conversion is assumed between lux and prototype brightness %, or between dBA and optional-audio volume %. A reading alone does not establish that the patient is stressed, comfortable, sleeping, or clinically abnormal.', 'role': 'measurement_convention', 'context_fields': ['lighting', 'sound'], 'condition': 'always'}
POLICY={"brightness_step":10,"sound_volume":20,"sound_duration_minutes":15,
        "night_start":19,"night_end":7,"comparison_tolerance":0.01,
        "meaning":"simulation settings, not clinically validated doses"}
METRICS={"hr_bpm":"bpm","rmssd_ms":"ms","eda_us":"microsiemens"}
TONES=["WARM","NEUTRAL","COOL"]
LIGHT_ACTIONS=["INCREASE","DECREASE","NO_CHANGE"]
SOUNDS=["PREFERRED_CALMING_MUSIC","NATURE_SOUNDS","WHITE_NOISE","NONE_SILENCE","NO_CHANGE"]

EVIDENCE_VERSION='E2_final_curated_18research_plus_E04_E06_and_E36_v1'
EVIDENCE_REVIEW_STATUS='FROZEN BEFORE EXPERIMENT 2 MODEL RUNS'
write_frozen(OUT/'evidence_table.json',{'version':EVIDENCE_VERSION,'status':EVIDENCE_REVIEW_STATUS,
    'entries':EVIDENCE,'environment_measurement_convention':ENVIRONMENT_MEASUREMENT_CONVENTION,'simulation_policy':POLICY})
review_cols=['id','source','study_design','exposure','finding','limitations','role','context_fields','lighting_actions','sounds','condition']
review_rows=[]
for e in EVIDENCE:
    review_rows.append({k:e.get(k) for k in review_cols})
pd.DataFrame(review_rows).to_csv(OUT/'evidence_review.csv',index=False)
print('Evidence frozen:',sum(e['kind']=='research' for e in EVIDENCE),'research +',sum(e['kind']=='prototype_rule' for e in EVIDENCE),'prototype rules + E36 measurement convention')


In [ ]:
# 5 · Embedded frozen 12-case synthetic Experiment 2 input snapshot
# Source workbook SHA256: 9a6aca612b5d6bb91ccf5bf9e849a34144803965da2ae0464dce30989af19ae3
_CASE_BYTES=zlib.decompress(base64.b64decode(''.join([
    'eNrtnW1v2zgSgP8KIaDfHJ/txqmTfMom7jZAaxdxiuKwWAi0Rce8SqJKUkmNvfz3myH15thp0p7tDbbzJbHFt5nhkJxHlOg//gpm3IhQRsFJMOyFnU43aAXZ'
    'Aq7BBfEtE1omIrW9cKZSq1Uciyg0y9QuhJWzsNuD3FbaGHOPb4VO5c3CMhMLkbH/skmeZUpbeSuYSG+lVinWhQmWT2PBssXSSBWrmyU2qtWtSHk6w7quhBFc'
    'zxZCH/DcLpQWEauaZQtlMml5zMwMSmip2uzdVYtdfZhMLlpseHHWYlMRGRkJJuM4T6SrlvE0YjyZSpQh+u2Mce3TjdXcCXnL41wYNlea1eoyo+b2DvMKTIec'
    'Kj1lqbLs83ByduFqxW8ZpGDNamqEvnXZTBv0ioSRN2kY86mITXDyVzBFgxtpfVXrhoMyDWuh4SNZ5KwN6nqptN5qHmfb4L6ZAZtd6HCaJfhJi7nQwhn6zWG7'
    '0wpmuYbvFr4e4dc8lfA5wNytOnP4NeextFBX8Pt4fBFUxdYT6jJ3Mo3UXQg9kFsB2vfrUpuSVJJx7cQ/sToXrcCoXDeb9taLZSrCG57VhUFqkycJ18swEeAv'
    '0ZM+1D+AwuA2rCgHYpfeFHIDF7PCnmeTyacPw4sQdQsn/x5dvxteX56Hb8dX4WT89vrz2dUwvB5Ori9Hv0MdddEQnOJpV864xAtz6NIDr00hkPwBR9zkepVf'
    'zsBacsbjeIn+LSNuRdQO7sE/dGJMFCbmgU/0eys+0e80fAIy/9Ndws0iv7BXiIiH+UOf6LYHDZfoto9rj5AzrYwUMFf9830jETzF9eWXdY97tHpqxTeLxo0V'
    'ZAktRAggcu/1SbcDytwoHsPXcinTAlcsiBr4DNYt3+9Xleaw6M9ElGuoYM5jA11YthxW7QRnUR5bsH2E1w9AmYjJtBSfR4m0Fq0BeqtqJW1oheqywo1Y1WCb'
    'feQybbG5gDJOby24Uamrh6fLOjjBVtlswdMb4UIGtI8wuMpKsxAoC7MLaepgBPXy0pXd+bEQ9o6n1jCr/ELfAmnQ9Q1c1wmLUXCZ3rSYVjl8EGD83ECP3i2k'
    'xWalEU66orALZ2KjwA/SyLAbaABiKg1a/SvlFlSEvs8xhc9mInNRQZt9btQFMoNDQXFtoMcXSkKMJOdMOU8EnXkeSYW5vPJRm41qQzZCFDQPDMUYWraYXYuZ'
    '0hE6DAwPy+fz0gzvCw2LCkGyOFZ32HdaJazXeYWmGfRfnTqDtFgqcgjNYuY8WsWQmtaF2mxcCur0rCrLM6ym13/FbhUEf95mxcUOK+YA5xSVMzAoL/QdX7Yf'
    'BF8w4A2k42cfu/mwMmyElWGcfwv78CXktzcwNw5wuSzCzNAZOoym3OeIufganBy6BRbiwpmW5YRxIRPvBVqBKe6kXTDQhhkreLRkzucXeXJa9vKDDoaPPNYu'
    'K3TDEk3MravAm8BpCyNR4DgGs3I9XUIJaA9C1Ptq3qyH6MrkH0yqmac55VQ5mJ+b20FjiWiU4dF/+Axdplm4dKO6aKaMdRNBMIIBBpdBXPgD6uIcBFNbBEH9'
    'EgJd57TgV6hKRQLFbOFcrsITnA7AHNUorOJhN3ZWPTjhqczyuJwHEQ1qERiYy8fxpyxSbgaQKWgPooCdUR7XbTMO49XLt2n2KKIuyOlcSeNoAIcGx+o5j0D3'
    'BjFhcfgQuD5JcV0anV1/gtViMv40upjAdd+j4GguLgPfhGEHHV4vbl2oDKP/qn+c55azS1iMvbAYLo2lM42+nwgTQwgzU6hAcvD9uJqzfVMa0bAQ+vO7y+th'
    'OBpfToZBM92N4ODkD6/kn24BWMiptI201avexfHyfTGbrGgDS9AXlLOQpDYq2gMM64zUvMq/BSeDvhujXscHMrWC0fDT9dXZe/h0Ph6/RyG9+g9sUhavBFzr'
    'qY9Xw7fDqysIA87P3n+ABS/88GlyeQ4pTfNA/SBUWHZrz8mGV2Bict5Y96zXJlW+TcyAJmisoSgZ+Nga1Pf2AfXnfpUgqt8F1RfG/TmsHwwI6wnrV7H+9YCw'
    'nrB+Bet7+I2wnrCesJ6wnrCesJ6wnrCesP7lYv3rLWL9hTQ6z2ivfmtUXxuU9uoJ6mmvnqCe9uoJ6gnqCeoJ6l8M1HeepPqjzjrVXyEVgvE8gHheTB1J6sK5'
    'XbefgrtBqvN5sDYYWssIEzMhZosWICcaK1YotR/O4msuMxd3F+bDOwDQsxiAQ/UZ+gMY7qAw1orrEPG/AOLv9xvE7ziTiJ+If3vEf7gP4qeN/B0iP23kE/PT'
    'Rj4xP23kE/MT8xPzE/MT8xPzE/MT8xPzb2b+/k8z/wVf4uIPs/UXMc9jtCG9lf9c7N9gvH2+mT9YpbxBh8j/lyd/j/qVTxzSbj+Rf6/9hsifyP975N/tnLxu'
    'kD/HFW2b5F/DPuA5B7RY8KmEbnaSu4ecW8z4Z5ShpyAGcMpY7R5otvgEM4bBwIdCQwjg4iIXhkfAe4i0GzCwSboZFgbTFncKDCC5cREN0BqGL7JA3SeZ33Jg'
    'CzSO66lVNep7ASX01rcDUJtYfsEe5gmU4NDNPE5QjSQ3CFHNh7bb7Bp08wnI7l6LuPGId0FJpVHvVB77Bpi0zoHAsdCWc+ZCapY53twP/R93tkP/nQ303+3v'
    'nP77vafpv7+B/n9zEIIe6WLC+ml+MCU0pItn+cuRtPpgPj2R/2JY/ajJ6jUrluT7OPSVhDd4jNs7fz+3Py79BoYvlf+7MP6484IxvvM4xnf7P4nxR/vCeNq+'
    '3zHH/9gW/gOQ9zv6BPIE8g2QPyaQJ5B/API9AnkCeQJ5AnkCeQJ5AnkCeQL5lwHyb7YM8vTm/Q44futv39N+PGE87ccTxtN+PGE8YTxhPGH8VjC+//SxeoPH'
    'H8bP09zkbuxFMnGSR+tR4ilT0H36Dl+p0CJ2QSyU+JpLYYnlX8Ixef3Hj8kjkCeQ3xXID/YF8rQjv2OSpx15QnnakSeUpx15QnlCeUJ5QnlCeUJ5QnlC+V8E'
    '5Y9/GuU/wmhVUMATMaxz6HowJRhLr8o/F+gft+E+35j3Tl9N3MddwnrC+lWf6BHWE9b38ET8+vjEpkcQ1hPWu7PyOieH/R2elQdodbPpsDwI1BB7GJ/jCWMc'
    'xE8PwBv1DXjUkhVLLCpdoy2usg6V3VPMm9mOnTd53gCk+ljCisRViZnLk/MMLMsxMMrpxsjfsAW/xYzM9TP0iUOF59wDcCEVtvrIoXrT3ELlBvAEzQY1Pzjn'
    'Dhl+ymdfdgzjh8Upd/4fyIoehYwIkKkduvm47FYa6e5hLD20I6uX2I4BN/RlvFxD9aw4Gi5CEPO3GEpAr6gdbVTzujdM86YFGglsgxbK7f/P7UfP4PajdW7/'
    '7G5UJCpCBxIFtlehX/PX7g5iuBx/77l6+uW6l8bxh9/h+NF4BAx4+X44Oh824P2xE+22ye7FvPoYvPuy6+z+QOIfO9GuguRNzL6GxKtn5z0GyD+F+oebj8N7'
    'FurvmfG7/e0zvju8dp+MT7v2+4H8H9u8f0D5Xf8bm4T5hPkNp+gT5hPmr2B+nzCfMJ8wnzCfMJ8wnzCfMJ8w/4Vifnc3mE9v2e+O8rf+sj1t5RPj01Y+MT5t'
    '5RPjE+MT4xPjP4vx3zzN+P42wCrjV1b0WFd0J8qep8VPC3rR52g6lLsg+UTBZAvGKl5PgALeDjBGgICNf0ml8Xt2xPzE/MT8xPxrzN/bM/PT1v5+oJ+29gn7'
    'aWufsJ+29gn7CfsJ+wn7CfsJ+wn7Cft/Qez/83+05TBS'
])))
assert hashlib.sha256(_CASE_BYTES).hexdigest()=='da7a13dc7d6cac3c80979cb61f8d33eb5fb307e388ecb25b70e586612a054105'
CASES=json.loads(_CASE_BYTES)
SMOKE_CASE=copy.deepcopy(CASES[0]);SMOKE_CASE['case_id']='E2_PILOT_001';SMOKE_CASE['phase']='development_smoke_copy_not_formal'
assert len(CASES)==12 and [c['case_id'] for c in CASES]==CASE_IDS

def check_case(c):
    assert c['case_id'] in CASE_IDS or c['case_id']=='E2_PILOT_001'
    assert c['context']['goal'] in ['sleep','awake']
    assert c['context']['reported_activity'] in ['RESTING','MOVING','UNKNOWN']
    assert re.fullmatch(r'(?:[01]\d|2[0-3]):[0-5]\d',c['context']['local_time'])
    assert set(c['physiology'])==set(METRICS)
    for k,x in c['physiology'].items():
        assert x['unit']==METRICS[k]
        assert x['reference_quality']==x['current_quality']=='GOOD'
        assert x['reference_window_minutes']==x['current_window_minutes']==5
        assert x['comparable'] is True
        assert type(x['reference']) in [int,float] and type(x['current']) in [int,float]
    em=c['context']['environment_measurements']
    assert em['bedside_illuminance_lux_5min_avg']>=0 and em['ambient_noise_dba_5min_laeq']>=0 and em['description']
    s,r,p=c['staff'],c['room'],c['preferences']
    assert 0<=s['brightness_min']<=r['brightness']<=s['brightness_max']<=100
    assert r['tone'] in s['allowed_tones'] and set(s['allowed_tones'])<=set(TONES)
    assert r['sound'] in SOUNDS and r['sound']!='NO_CHANGE'
    assert 0<=r['volume']<=100 and r['remaining_minutes']>=0
    if r['sound']=='NONE_SILENCE':assert r['volume']==0 and r['remaining_minutes']==0
    assert set(s['allowed_sounds'])<=set(SOUNDS)-{'NO_CHANGE','NONE_SILENCE'}
    assert p['preferred_sound'] is None or p['preferred_sound'] in SOUNDS[:-1]
    assert p['preferred_tones'] is None or set(p['preferred_tones'])<=set(TONES)
    assert set(p['prohibited_tones'])<=set(TONES)
    assert set(p['prohibited_sounds'])<=set(SOUNDS)-{'NO_CHANGE'}
    assert all(type(v) is bool for v in [p['lighting_changes_allowed'],p['sound_changes_allowed'],s['lighting_locked'],s['sound_allowed'],s['no_sound_during_procedure'],c['context']['procedure']])

for c in CASES+[SMOKE_CASE]:check_case(c)
assert sum(c['design_labels']['environment_condition']=='Supportive' for c in CASES)==6
assert sum(c['design_labels']['environment_condition']=='Disruptive' for c in CASES)==6
assert sum(c['design_labels']['physiology_condition']=='Stable' for c in CASES)==6
assert sum(c['design_labels']['physiology_condition']=='Changed' for c in CASES)==6
# Factor isolation: physiology repeats across environment for each base+physiology condition; room/environment repeats across physiology for each base+environment condition.
for base in sorted({c['design_labels']['base_situation'] for c in CASES}):
    subset=[c for c in CASES if c['design_labels']['base_situation']==base]
    for pc in ['Stable','Changed']:
        pair=[c for c in subset if c['design_labels']['physiology_condition']==pc]
        assert len(pair)==2 and encode(pair[0]['physiology'])==encode(pair[1]['physiology'])
    for ec in ['Supportive','Disruptive']:
        pair=[c for c in subset if c['design_labels']['environment_condition']==ec]
        assert len(pair)==2
        assert encode(pair[0]['room'])==encode(pair[1]['room'])
        assert encode(pair[0]['context']['environment_measurements'])==encode(pair[1]['context']['environment_measurements'])
# Save the exact model input snapshot; no nurse data is present here.
p=OUT/'experiment2_case_inputs.json'
if p.exists():assert p.read_bytes()==_CASE_BYTES,'Case snapshot changed; choose a new RUN_LABEL.'
else:p.write_bytes(_CASE_BYTES)
write_frozen(OUT/'development_smoke_case.json',SMOKE_CASE)
print('12 Experiment 2 cases verified: 3 bases × 2 environments × 2 physiology states. Nurse data not loaded.')


In [ ]:
# 6 · Exact development workflow tools, unchanged
# 6 · Four bounded tools: no arbitrary code execution

def comparisons(c):
    out={}
    for k,p in c['physiology'].items():
        usable=(p['reference'] is not None and p['current'] is not None and
                p['reference_quality']==p['current_quality']=='GOOD' and p['comparable'])
        delta=p['current']-p['reference'] if usable else None
        pct=100*delta/p['reference'] if usable and p['reference']!=0 and k!='eda_us' else None
        out[k]={'status':'AVAILABLE' if usable else 'UNAVAILABLE',
                'absolute_change':None if delta is None else round(delta,6),
                'percent_change':None if pct is None else round(pct,6)}
    return out

def tool(name, args, c):
    if name=='calculate_changes': return comparisons(c)
    if name=='check_data_quality':
        return {k:{**{f:p[f] for f in ['reference_quality','current_quality','reference_window_minutes','current_window_minutes','comparable']},
                   'comparison_available':comparisons(c)[k]['status']=='AVAILABLE'} for k,p in c['physiology'].items()}
    if name=='check_activity_context': return copy.deepcopy(c['context'])
    if name=='retrieve_evidence':
        ids=args['evidence_ids']; known={e['id'] for e in EVIDENCE}
        if not set(ids)<=known: raise ValueError('Unknown evidence ID')
        return [e for e in EVIDENCE if e['id'] in ids]
    raise ValueError('Undeclared tool: '+name)

def spec(name, description, properties=None, required=None):
    return {'type':'function','function':{'name':name,'description':description,
        'parameters':{'type':'object','properties':properties or {},'required':required or [],'additionalProperties':False}}}
TOOLS=[spec('calculate_changes','Calculate descriptive HR, RMSSD and EDA changes from the supplied case; no arguments.'),
       spec('check_data_quality','Inspect recorded quality and comparability, without diagnosing signal validity.'),
       spec('check_activity_context','Return patient context and separate source activity exactly as supplied.'),
       spec('retrieve_evidence','Optionally re-read approved evidence summaries by evidence_ids. The complete curated summaries are already in evidence_table, so retrieval is not required before citing them. Returns the same roles, applicable actions and limitations; it does not retrieve full papers.',
            {'evidence_ids':{'type':'array','items':{'type':'string','enum':[e['id'] for e in EVIDENCE]},'minItems':1}},['evidence_ids'])]
TOOL_SCHEMAS={x['function']['name']:x['function']['parameters'] for x in TOOLS}


In [ ]:
# 7 · Exact development output JSON schema and deterministic validator, unchanged
# 7 · Output contract and deterministic simulation

def obj(props): return {'type':'object','properties':props,'required':list(props),'additionalProperties':False}
def enum(values): return {'type':'string','enum':values}
def text_schema(): return {'type':'string','minLength':1,'pattern':r'\S'}
RESEARCH_IDS={e['id'] for e in EVIDENCE if e['kind']=='research'}
POLICY_IDS={e['id'] for e in EVIDENCE if e['kind']=='prototype_rule'}
def ids_schema(ids): return {'type':'array','uniqueItems':True,'items':enum(sorted(ids))}
COMPARISON_SCHEMA=obj({'status':enum(['AVAILABLE','UNAVAILABLE']),
                      'absolute_change':{'type':['number','null']},'percent_change':{'type':['number','null']}})
OUTPUT_SCHEMA=obj({
 'physiological_interpretation':text_schema(), 'physiology_evidence_ids':ids_schema(RESEARCH_IDS), 'certainty':enum(['LOW','MEDIUM','HIGH']), 'certainty_rationale':text_schema(),
 'activity_assessment':enum(['RESTING','MOVING','UNKNOWN']),
 'clinical_review':obj({'recommended':enum(['YES','NO']),'rationale':text_schema(),'evidence_ids':ids_schema(RESEARCH_IDS)}),
 'comparisons':obj({k:COMPARISON_SCHEMA for k in METRICS}),
 'lighting':obj({'action':enum(LIGHT_ACTIONS),'tone':enum(TONES),'rationale':text_schema(),'evidence_ids':ids_schema(RESEARCH_IDS),'policy_ids':ids_schema(POLICY_IDS)}),
 'sound':obj({'option':enum(SOUNDS),'rationale':text_schema(),'evidence_ids':ids_schema(RESEARCH_IDS),'policy_ids':ids_schema(POLICY_IDS)}),
 'limitations':{'type':'array','items':text_schema(),'minItems':1}})

def settings_for(answer,c):
    r=c['room']; s=c['staff']; action=answer['lighting']['action']
    direction={'INCREASE':1,'DECREASE':-1,'NO_CHANGE':0}[action]
    # This is the declared action-to-setting mapping, not repair of invalid model output.
    brightness=min(s['brightness_max'],max(s['brightness_min'],r['brightness']+direction*POLICY['brightness_step']))
    option=answer['sound']['option']
    if option=='NO_CHANGE': sound,vol,dur=r['sound'],r['volume'],r['remaining_minutes']
    elif option=='NONE_SILENCE': sound,vol,dur=option,0,0
    else: sound,vol,dur=option,min(POLICY['sound_volume'],s['max_volume']),min(POLICY['sound_duration_minutes'],s['max_duration_minutes'])
    return {'brightness':brightness,'tone':answer['lighting']['tone'],'sound':sound,'volume':vol,'duration_minutes':dur}

def evidence_target_for(field, declared_choice, c, cfg):
    """Return the environmental state/direction that evidence must support.

    Sound NO_CHANGE is state-aware: it means retain the current room sound, so
    evidence is checked against cfg['sound'] (which equals c['room']['sound']).
    Lighting remains direction-based in this experiment because the frozen
    evidence table maps lighting research to INCREASE/DECREASE/NO_CHANGE.
    """
    if field=='sound' and declared_choice=='NO_CHANGE':
        return cfg['sound']
    return declared_choice

def validate_answer(a,c,available_research_ids=None):
    errors=[f'SCHEMA {list(e.path)}: {e.message}' for e in Draft202012Validator(OUTPUT_SCHEMA).iter_errors(a)]
    if errors: return errors,None
    # Fail closed unless research availability is explicitly supplied by the workflow.
    available=set(available_research_ids or [])
    cited=set(a['physiology_evidence_ids']) | set(a['clinical_review']['evidence_ids'])
    for field in ['lighting','sound']: cited.update(a[field]['evidence_ids'])
    # Also catch IDs named inside prose, not just the structured citation lists.
    prose=[a['certainty_rationale'],a['physiological_interpretation'],a['clinical_review']['rationale'],a['lighting']['rationale'],a['sound']['rationale'],*a['limitations']]
    inline=set(re.findall(r'\bE[0-9]+\b',' '.join(prose)))
    for eid in sorted(inline-RESEARCH_IDS-POLICY_IDS): errors.append('UNKNOWN_CITATION '+eid)
    cited.update(inline & RESEARCH_IDS)
    for eid in sorted(cited-available): errors.append('RESEARCH_NOT_AVAILABLE '+eid)
    for eid in a['physiology_evidence_ids']:
        entry=next(e for e in EVIDENCE if e['id']==eid)
        if 'physiology' not in entry['context_fields']: errors.append('EVIDENCE_FIELD_MISMATCH physiology/'+eid)
    for eid in a['clinical_review']['evidence_ids']:
        entry=next(e for e in EVIDENCE if e['id']==eid)
        if 'physiology' not in entry['context_fields']: errors.append('EVIDENCE_FIELD_MISMATCH clinical_review/'+eid)
    truth=comparisons(c)
    for k in METRICS:
        x=a['comparisons'][k]; expected=truth[k]
        if x['status']!=expected['status']: errors.append('COMPARISON_STATUS '+k)
        for f in ['absolute_change','percent_change']:
            v=x[f]; t=expected[f]
            if (t is None and v is not None) or (t is not None and (v is None or not math.isfinite(v) or abs(v-t)>POLICY['comparison_tolerance'])):
                errors.append('COMPARISON_VALUE '+k+'/'+f)
    if a['activity_assessment']!=c['context']['reported_activity']: errors.append('ACTIVITY_MISMATCH')
    cfg=settings_for(a,c); s=c['staff']; p=c['preferences']; r=c['room']; l=a['lighting']; snd=a['sound']['option']
    if s['lighting_locked'] or not p['lighting_changes_allowed']:
        if l['action']!='NO_CHANGE' or l['tone']!=r['tone']: errors.append('LIGHTING_LOCK')
    if l['tone'] not in s['allowed_tones']: errors.append('TONE_RESTRICTION')
    if l['tone'] in p.get('prohibited_tones',[]): errors.append('PATIENT_TONE_PROHIBITION')
    if l['action']=='INCREASE' and cfg['brightness']<=r['brightness']: errors.append('NO_INCREASE_POSSIBLE')
    if l['action']=='DECREASE' and cfg['brightness']>=r['brightness']: errors.append('NO_DECREASE_POSSIBLE')
    active=cfg['sound']!='NONE_SILENCE'
    forbidden=not s['sound_allowed'] or (c['context']['procedure'] and s['no_sound_during_procedure'])
    if active and forbidden: errors.append('SOUND_RESTRICTION')
    if active and cfg['sound'] not in s['allowed_sounds']: errors.append('SOUND_NOT_ALLOWED')
    if cfg['sound'] in p['prohibited_sounds']: errors.append('PATIENT_SOUND_PROHIBITION')
    if active and (cfg['volume']<=0 or cfg['duration_minutes']<=0): errors.append('SOUND_WITHOUT_PLAYBACK')
    if cfg['volume']>s['max_volume'] or cfg['duration_minutes']>s['max_duration_minutes']: errors.append('PLAYBACK_LIMIT')
    changed=(cfg['sound'],cfg['volume'],cfg['duration_minutes'])!=(r['sound'],r['volume'],r['remaining_minutes'])
    if changed and not p['sound_changes_allowed']: errors.append('SOUND_CHANGE_PROHIBITED')

    # Preference requirements apply to the resulting sound, including a retained
    # current sound under NO_CHANGE.
    resulting_sound=cfg['sound']
    if resulting_sound in ['PREFERRED_CALMING_MUSIC','WHITE_NOISE'] and p['preferred_sound']!=resulting_sound:
        errors.append('PREFERENCE_REQUIRED_FOR_THIS_PROTOTYPE_SOUND')

    night=int(c['context']['local_time'][:2])>=POLICY['night_start'] or int(c['context']['local_time'][:2])<POLICY['night_end']
    for field,declared_choice,allowed_key in [('lighting',l['action'],'lighting_actions'),('sound',snd,'sounds')]:
        target_choice=evidence_target_for(field,declared_choice,c,cfg)
        applicable=False
        research_action_support=False
        for eid in a[field]['evidence_ids']+a[field]['policy_ids']:
            e=next(e for e in EVIDENCE if e['id']==eid)
            if e.get('role')=='context_only':
                if field not in e['context_fields']: errors.append('EVIDENCE_FIELD_MISMATCH '+field+'/'+eid)
                continue  # contextual citation is permitted but cannot supply action support

            # For sound NO_CHANGE, evidence is checked against the retained
            # resulting sound state. For all other choices, target_choice equals
            # the declared action/option.
            if target_choice not in e[allowed_key]:
                errors.append(('POLICY_ACTION_MISMATCH ' if eid in POLICY_IDS else 'EVIDENCE_ACTION_MISMATCH ')+field+'/'+eid); continue

            condition_ok=True
            if e['condition']=='time_of_day':
                condition_ok=(declared_choice=='NO_CHANGE' or (declared_choice=='DECREASE' and night) or
                              (declared_choice=='INCREASE' and not night and c['context']['goal']!='sleep'))
            if e['condition']=='sound_permission':
                condition_ok=not forbidden
            if e['condition']=='sleep_goal':
                condition_ok=not forbidden and c['context']['goal'] in ['sleep','rest']
            if e['condition']=='preference_compatible' and target_choice in ['PREFERRED_CALMING_MUSIC','NATURE_SOUNDS','WHITE_NOISE']:
                condition_ok=p['preferred_sound']==target_choice
            if not condition_ok:
                errors.append('EVIDENCE_CONTEXT_MISMATCH '+field+'/'+eid)
            applicable=applicable or condition_ok
            if e.get('kind')=='research' and e.get('role')=='action_support':
                research_action_support=research_action_support or condition_ok

        if not applicable:
            errors.append('NO_APPLICABLE_SUPPORT '+field)

        # A retained active optional sound is a substantive recommendation about
        # the resulting environment. It therefore requires at least one applicable
        # research action-support source for that retained sound. E05 may still be
        # cited as a prototype basis, but cannot substitute for research here.
        if field=='sound' and declared_choice=='NO_CHANGE' and target_choice!='NONE_SILENCE' and not research_action_support:
            errors.append('NO_RESEARCH_SUPPORT_RETAINED_SOUND '+target_choice)

    return errors, cfg if not errors else None

CITATION_ERROR_PREFIXES=('UNKNOWN_CITATION ', 'RESEARCH_NOT_AVAILABLE ',
    'EVIDENCE_FIELD_MISMATCH ', 'POLICY_ACTION_MISMATCH ', 'EVIDENCE_ACTION_MISMATCH ',
    'EVIDENCE_CONTEXT_MISMATCH ', 'NO_APPLICABLE_SUPPORT ', 'NO_RESEARCH_SUPPORT_RETAINED_SOUND ')

def assess_answer(answer,c,available_research_ids=None):
    errors,settings=validate_answer(answer,c,available_research_ids)
    schema_errors=[e for e in errors if e.startswith('SCHEMA ')]
    # Do not infer domain success from an output that could not be structurally checked.
    if schema_errors:
        return {'schema_valid':False,'calculation_valid':None,'action_compliant':None,
          'citation_contract_valid':None,'schema_errors':schema_errors,
          'calculation_errors':[],'action_errors':[],'citation_errors':[],
          'overall_valid':False,'errors':errors,'settings':None,'semantic_grounding':'NOT_ASSESSED'}
    citation=[e for e in errors if e.startswith(CITATION_ERROR_PREFIXES)]
    calculation=[e for e in errors if e.startswith(('COMPARISON_STATUS ','COMPARISON_VALUE '))]
    action=[e for e in errors if e not in citation+calculation]
    return {'schema_valid':True,'calculation_valid':not calculation,'action_compliant':not action,
      'citation_contract_valid':not citation,'schema_errors':[],
      'calculation_errors':calculation,'action_errors':action,'citation_errors':citation,
      'overall_valid':not errors,'errors':errors,'settings':settings,'semantic_grounding':'NOT_ASSESSED'}


In [ ]:
# 8 · Shared Experiment 2 instructions and frozen model-visible inputs
COMMON = 'You are a non-clinical Adaptive Hospital Room (AHR) research prototype. Recommend simulated lighting and optional sound for staff review, and separately state whether the supplied physiology/context suggests that human clinical review should be considered before attributing changes to the room.\nUse only supplied case facts and evidence. Never diagnose stress, disease, pain, fever, or relaxation from these signals alone. A clinical_review recommendation is advisory: YES does not diagnose an emergency or prohibit a room adjustment; NO does not rule out illness or replace normal clinical observation.\nSet clinical_review.recommended to YES only when the supplied case contains a case-specific observed physiological or clinical concern that warrants human assessment before attributing the finding to the room. Missing pain or fever information, an unknown cause, limited measurements, or general uncertainty are not by themselves sufficient reasons for YES. Do not use clinical_review=YES merely as a precautionary response to missing information.\nThe clinical-review flag is separate from the environmental recommendation: a case may warrant clinical review without requiring a different lighting or sound action, and vice versa.\nFor the clinical-review decision, evaluate the supplied synthetic physiology as the observations of the hypothetical patient described in the scenario. The fact that the values are researcher-authored synthetic data must not by itself determine YES or NO.\nThe physiology values are researcher-authored synthetic paired five-minute summaries. Report signed current-minus-reference changes. HR/RMSSD percent = 100*(current-reference)/reference. EDA percent is always null. These comparisons are descriptive and do not establish cause.\nThe environmental lux and dBA values are researcher-authored five-minute room measurements. They are environmental context, not physiological thresholds. There is no validated direct conversion from lux to prototype brightness %, or from dBA/LAeq to optional-audio volume %. Do not invent thresholds or conversions.\nPhysiology and environment are manipulated independently in this study. Do not infer that the room caused a physiological change merely because they appear in the same scenario.\nRespect explicit staff restrictions, patient permissions, opt-outs and prohibitions. Structured constraints are authoritative. Patient preferences are preferences unless recorded as prohibitions/opt-out.\nBrightness/volume/duration are assigned by the supplied prototype policy; do not generate those numbers. Lighting action describes brightness direction; tone is a separate choice. A lighting lock also locks tone.\nNONE_SILENCE stops optional playback; it does not silence ward noise, monitor tones or staff conversation. Do not claim optional audio necessarily masks ambient noise.\nBefore choosing sound, inspect the current sound, volume and remaining duration. NO_CHANGE means retain the resulting current sound state. If an active sound is retained, cite at least one applicable action-support research source for that retained sound; source conditions still apply. If current sound is silence, E05 may support retaining silence. Do not change sound merely to make a citation fit.\nEvery lighting/sound choice needs an applicable action-support research source or optional prototype action basis. Context-only sources may qualify reasoning but cannot alone justify an action. Research findings do not prove individual benefit.\nUse evidence_ids only for supplied research entries and policy_ids only for E04-E06 prototype rules. E36 is a non-citable measurement convention supplied separately.\nList research used for physiological interpretation in physiology_evidence_ids. clinical_review.evidence_ids may cite physiology-context research; [] is allowed when the flag is based only on explicit case facts and uncertainty.\nCertainty is qualitative confidence that the whole environmental recommendation fits the supplied case, not diagnostic certainty. HIGH requires unusually clear recorded facts/constraints; correct arithmetic or a validation pass alone does not justify HIGH.\nReturn one JSON object exactly matching OUTPUT_SCHEMA, in assistant content, with no markdown and no extra fields. Provide concise evidence-grounded rationales and limitations, not private step-by-step reasoning.\n'


def messages_for(c,arm='agent'):
    require(arm=='agent','Experiment 2 final notebook runs the agentic workflow only')
    research=[e for e in EVIDENCE if e['id'] in RESEARCH_IDS]
    return [{'role':'system','content':COMMON+'\nYou may call any declared tools as needed, in any order, or answer directly. The complete curated research summaries are already supplied in evidence_table; retrieve_evidence is optional. Use native tool calls only for declared functions; JSON is a format, not a tool.'},
            {'role':'user','content':encode({
              'case':c,
              'simulation_policy':POLICY,
              'mandatory_constraints':{'staff':copy.deepcopy(c['staff']),
                'patient_permissions_and_prohibitions':copy.deepcopy(c['preferences']),
                'procedure_active':c['context']['procedure']},
              'optional_action_bases':[e for e in EVIDENCE if e['id'] in ['E04','E05']],
              'calculation_conventions':[e for e in EVIDENCE if e['id']=='E06'],
              'environment_measurement_conventions':[ENVIRONMENT_MEASUREMENT_CONVENTION],
              'OUTPUT_SCHEMA':OUTPUT_SCHEMA,
              'evidence_table':research
            })}]


In [ ]:
# 9 · Luna-only OpenRouter transport and spending guards (from successful development)
# 9 · Bounded transport; no network is used merely by defining this cell
class RunStopped(Exception): pass

HALT_STATUSES={'PROVIDER_FAILURE','GLOBAL_BUDGET_STOP','BUDGET_STOP','REQUEST_LIMIT','REQUEST_COST_STOP',
 'REPORTED_COST_EXCEEDED_LOCAL_ESTIMATE','UPSTREAM_IDENTITY_MISMATCH','EMPTY_CHOICES','MALFORMED_RESPONSE','RUNNING'}
# INPUT_SIZE_STOP is a terminal outcome for one workflow, not a global provider/budget halt.
# Compact correction was designed to keep repair requests below the unchanged 100k local guard.
print('Experiment 2 transport guards loaded; INPUT_SIZE_STOP will not halt unrelated NOT_RUN workflows.')

def provider_error(raw,http_status):
    if not isinstance(raw,dict): return {'code':http_status,'message':'Non-object response','transient':False}
    choices=raw.get('choices'); choice=choices[0] if isinstance(choices,list) and choices and isinstance(choices[0],dict) else {}
    error=raw.get('error') or choice.get('error')
    if error is None and choice.get('finish_reason')!='error' and http_status==200: return None
    error=error if isinstance(error,dict) else {'message':str(error or 'Provider generation error')}
    try: code=int(error.get('code',http_status))
    except (TypeError,ValueError): code=http_status
    return {'code':code,'message':str(error.get('message','Provider error')),
            'transient':code in [429,500,502,503,504]}

def price(value):
    try: result=float(value)
    except (TypeError,ValueError): raise ValueError('Missing/invalid endpoint price')
    require(math.isfinite(result) and result>=0,'Invalid endpoint price')
    return result

def eligible_endpoints(data):
    require(isinstance(data,dict) and data.get('id')==MODEL,'Endpoint catalogue model mismatch.')
    approved=[]
    required={'tools','tool_choice','max_tokens'}
    if SEND_TEMPERATURE: required.add('temperature')
    if REASONING_EFFORT is not None: required.add('reasoning')
    for e in data.get('endpoints',[]):
        try:
            tag=e.get('tag'); require(isinstance(tag,str) and tag,'Missing endpoint tag')
            require(e.get('provider_name')==PROVIDER,'Provider mismatch')
            require(not any(x in tag.lower() for x in ['/flex','/priority','/fast']), 'Nonstandard provider route excluded')
            require(required<=set(e.get('supported_parameters') or []),'Missing parameter capability')
            require((e.get('supports_tool_choice') or {}).get('auto') is True,'Auto tool choice not explicitly supported')
            require(e.get('status')==0,'Endpoint not listed as enabled')
            require(isinstance(e.get('max_completion_tokens'),int) and e['max_completion_tokens']>=MAX_OUTPUT_TOKENS,'Output cap unsupported')
            require(isinstance(e.get('context_length'),int) and e['context_length']>=MAX_OUTPUT_TOKENS+MAX_ESTIMATED_INPUT_TOKENS,'Context allowance insufficient')
            prices=e['pricing']
            require(price(prices['prompt'])*1e6<=MAX_INPUT_PRICE_PER_M+1e-9,'Input price too high')
            require(price(prices['completion'])*1e6<=MAX_OUTPUT_PRICE_PER_M+1e-9,'Output price too high')
            require(price(prices.get('request',0))==0,'Per-request fee unsupported')
            require(price(prices.get('internal_reasoning',prices['completion']))*1e6<=MAX_OUTPUT_PRICE_PER_M+1e-9,'Reasoning price too high')
            approved.append(e)
        except (ValueError,KeyError,TypeError): continue
    require(approved,'No explicitly compatible endpoint within the configured prices/cap. No generation sent.')
    # Freeze one and only one endpoint. Do not allow fallback or cross-provider routing.
    approved.sort(key=lambda e:(price(e['pricing']['completion']),price(e['pricing']['prompt']),str(e['tag'])))
    return [approved[0]]

def master_used_usd():
    # Includes known charges and conservatively reserved unknown-cost requests.
    total=0.0
    for name in PROFILES:
        p=MASTER_ROOT/name/'cost_ledger.json'
        if not p.exists(): continue
        ledger=json.loads(p.read_text())
        require(isinstance(ledger,list),'Invalid cross-model cost ledger: '+name)
        for item in ledger:
            amount=item.get('accounted_usd') if isinstance(item,dict) else None
            require(type(amount) in (float,int) and math.isfinite(amount) and amount>=0,'Invalid cost entry: '+name)
            total+=amount
    return total

class Router:
    def __init__(self,key,out,endpoints=None):
        self.key=key;self.out=Path(out);self.out.mkdir(parents=True,exist_ok=True);self.last=0
        self.path=self.out/'cost_ledger.json';self.endpoints=endpoints or []
        self.reload()
    def reload(self):
        self.ledger=json.loads(self.path.read_text()) if self.path.exists() else []
        require(isinstance(self.ledger,list),'Corrupt ledger')
        for x in self.ledger:
            require(isinstance(x,dict) and type(x.get('accounted_usd')) in [int,float] and math.isfinite(x['accounted_usd']) and x['accounted_usd']>=0,'Corrupt ledger; stop for review')
    def get(self,path):
        r=requests.get('https://openrouter.ai/api/v1/'+path,headers={'Authorization':'Bearer '+self.key},timeout=30)
        r.raise_for_status();return r.json()
    def used(self): return sum(x['accounted_usd'] for x in self.ledger)
    def payload(self,messages,use_tools):
        require(self.endpoints,'Endpoint preflight must complete before generation.')
        require(len(self.endpoints)==1 and self.endpoints[0]['provider_name']==PROVIDER,'Exactly one verified provider endpoint must be pinned.')
        provider={'require_parameters':True,'max_price':{'prompt':MAX_INPUT_PRICE_PER_M,'completion':MAX_OUTPUT_PRICE_PER_M,'request':0},
                  'only':[self.endpoints[0]['tag']],'allow_fallbacks':False}
        payload={'model':MODEL,'messages':messages,'max_tokens':MAX_OUTPUT_TOKENS,'provider':provider}
        if SEND_TEMPERATURE: payload['temperature']=TEMPERATURE
        if REASONING_EFFORT is not None: payload['reasoning']={'effort':REASONING_EFFORT}
        if use_tools: payload.update(tools=TOOLS,tool_choice='auto')
        return payload
    def reservation(self,payload):
        # Conservative local upper bound: a UTF-8 BPE token cannot consume fewer than one input byte.
        # v2 accidentally doubled the byte count, causing false INPUT_SIZE_STOP outcomes after tool calls/corrections.
        estimate=len(encode(payload).encode('utf-8'))+4096
        return estimate,(estimate*MAX_INPUT_PRICE_PER_M+MAX_OUTPUT_TOKENS*MAX_OUTPUT_PRICE_PER_M)/1e6
    def call(self,messages,use_tools,tag):
        payload=self.payload(messages,use_tools);attempts=[]
        for retry in range(TRANSIENT_RETRIES+1):
            self.reload()
            estimate,reserve=self.reservation(payload)
            if len(self.ledger)>=MAX_HTTP_REQUESTS:return {'error':'REQUEST_LIMIT','attempts':attempts}
            if estimate>MAX_ESTIMATED_INPUT_TOKENS:return {'error':'INPUT_SIZE_STOP','attempts':attempts}
            if reserve>MAX_REQUEST_RESERVATION_USD+1e-12:return {'error':'REQUEST_COST_STOP','attempts':attempts}
            if self.used()+reserve>RUN_BUDGET_USD+1e-12:return {'error':'BUDGET_STOP','attempts':attempts}
            if master_used_usd()+reserve>GLOBAL_BUDGET_USD+1e-12:return {'error':'GLOBAL_BUDGET_STOP','attempts':attempts}
            wait=max(0,PACE_SECONDS-(time.monotonic()-self.last))
            if wait:time.sleep(wait)
            uid=uuid.uuid4().hex
            item={'id':uid,'tag':tag,'time':stamp(),'reserved_usd':reserve,'accounted_usd':reserve,'cost_known':False}
            self.ledger.append(item);save_json(self.path,self.ledger)
            save_json(self.out/'requests'/f'{uid}_request.json',payload)
            started=time.monotonic();self.last=started;r=None
            try:
                r=requests.post('https://openrouter.ai/api/v1/chat/completions',headers={'Authorization':'Bearer '+self.key,'Content-Type':'application/json'},json=payload,timeout=REQUEST_TIMEOUT)
                try: raw=r.json()
                except ValueError:raw={'unparsed_body':r.text[:20000]}
                status=r.status_code
            except requests.RequestException as exc:
                raw={'transport_error':type(exc).__name__};status=0
            if not isinstance(raw,dict):raw={'malformed_body':raw}
            failure=provider_error(raw,status)
            attempt={'provider_error':failure,'id':uid,'http_status':status,'latency_seconds':time.monotonic()-started,'raw':raw}
            attempts.append(attempt);save_json(self.out/'requests'/f'{uid}_response.json',attempt)
            usage=raw.get('usage') if isinstance(raw.get('usage'),dict) else {};cost=usage.get('cost')
            if type(cost) in [int,float] and math.isfinite(cost) and cost>=0:item.update(accounted_usd=float(cost),cost_known=True)
            item.update(http_status=status,provider=raw.get('provider'),generation_id=raw.get('id'));save_json(self.path,self.ledger)
            if item['accounted_usd']>reserve+1e-12 or self.used()>RUN_BUDGET_USD+1e-12 or master_used_usd()>GLOBAL_BUDGET_USD+1e-12:
                return {'error':'REPORTED_COST_EXCEEDED_LOCAL_ESTIMATE','attempts':attempts}
            if failure is None:
                choices=raw.get('choices')
                if not isinstance(choices,list) or not choices:return {'error':'EMPTY_CHOICES','attempts':attempts}
                choice=choices[0]
                if not isinstance(choice,dict) or not isinstance(choice.get('message'),dict):return {'error':'MALFORMED_RESPONSE','attempts':attempts}
                if raw.get('model')!=MODEL or raw.get('provider') not in {e['provider_name'] for e in self.endpoints}:
                    return {'error':'UPSTREAM_IDENTITY_MISMATCH','attempts':attempts}
                return {'choice':choice,'raw':raw,'attempts':attempts}
            if failure['transient'] and retry<TRANSIENT_RETRIES:
                try:delay=float(r.headers.get('Retry-After','15')) if r is not None else 15
                except (ValueError,TypeError):delay=15
                if not math.isfinite(delay) or delay>60:break
                time.sleep(max(1,delay));continue
            break
        return {'error':'PROVIDER_FAILURE','attempts':attempts}

CLIENT=None

def prepare_client():
    require(globals().get('OFFLINE_CHECKS_PASSED') is True,'Run offline checks before live execution.')
    try:
        from google.colab import userdata
        key=userdata.get('OPENROUTER_API_KEY')
    except Exception:
        from getpass import getpass
        key=os.environ.get('OPENROUTER_API_KEY') or getpass('OpenRouter API key (hidden): ')
    require(isinstance(key,str) and bool(key.strip()),'An OpenRouter key is required.')
    client=Router(key,OUT)
    data=client.get('models/'+MODEL+'/endpoints')['data']
    client.endpoints=eligible_endpoints(data)
    save_json(OUT/('endpoint_preflight_'+uuid.uuid4().hex+'.json'),{'checked':stamp(),'data':data,'approved_tags':[e['tag'] for e in client.endpoints]})
    print('SINGLE selected provider:',PROVIDER,'| verified endpoint:',[e['tag'] for e in client.endpoints])
    print('HQ-based provider filter only; data residency is NOT independently verified.')
    print('Local limits: $',RUN_BUDGET_USD,'per run; $',MAX_REQUEST_RESERVATION_USD,'reservation per request;',MAX_HTTP_REQUESTS,'HTTP attempts.')
    print('These are local safeguards, not guaranteed account billing caps. Unknown cost stays reserved.')
    return client
print('Client defined. No network or generation calls in this cell.')



In [ ]:
# 10 · Final deployable agentic workflow: initial tools -> compact correction -> bounded mechanical repair

def parse_final(content):
    if not isinstance(content,str) or not content.strip(): raise ValueError('EMPTY_FINAL_CONTENT')
    def unique(pairs):
        out={}
        for k,v in pairs:
            if k in out: raise ValueError('DUPLICATE_JSON_KEY '+k)
            out[k]=v
        return out
    return json.loads(content,object_pairs_hook=unique,
                      parse_constant=lambda x: (_ for _ in ()).throw(ValueError('NONFINITE_JSON')))

def evaluate_content(content,c,available_research):
    try:
        answer=parse_final(content)
        assessment=assess_answer(answer,c,available_research)
        return answer,assessment,assessment['errors'],assessment.get('settings')
    except Exception as exc:
        errors=[str(exc)]
        assessment={'schema_valid':False,'calculation_valid':None,'action_compliant':None,
                    'citation_contract_valid':None,'overall_valid':False,'errors':errors,
                    'schema_errors':errors,'calculation_errors':[],'action_errors':[],
                    'citation_errors':[],'settings':None,'semantic_grounding':'NOT_ASSESSED'}
        return None,assessment,errors,None

def compact_repair_messages(c,invalid_content,errors):
    research=[e for e in EVIDENCE if e['id'] in RESEARCH_IDS]
    obj={
      'case':c,'simulation_policy':POLICY,
      'mandatory_constraints':{'staff':copy.deepcopy(c['staff']),
        'patient_permissions_and_prohibitions':copy.deepcopy(c['preferences']),
        'procedure_active':c['context']['procedure']},
      'optional_action_bases':[e for e in EVIDENCE if e['id'] in ['E04','E05']],
      'calculation_conventions':[e for e in EVIDENCE if e['id']=='E06'],
      'environment_measurement_conventions':[ENVIRONMENT_MEASUREMENT_CONVENTION],
      'OUTPUT_SCHEMA':OUTPUT_SCHEMA,'evidence_table':research,
      'available_research_ids':sorted(RESEARCH_IDS),
      'original_invalid_output':invalid_content,'validation_errors':errors,
      'repair_instruction':'Correct the listed validation errors and return one complete JSON object. Reassess affected rationale/citations rather than simply deleting evidence. Preserve substantive choices unless the listed error or unchanged validator requires a change. Do not invent thresholds, diagnoses, benefits, facts or new evidence.'
    }
    sys=COMMON+'\nThis is a compact final-only correction of an already-generated answer. No tools are available. The validation errors are mechanical/contract feedback, not new patient evidence.'
    return [{'role':'system','content':sys},{'role':'user','content':encode(obj)}]

def mechanical_errors_only(errors):
    prefixes=('DUPLICATE_JSON_KEY ','SCHEMA ','COMPARISON_STATUS ','COMPARISON_VALUE ',
              'EMPTY_FINAL_CONTENT','NONFINITE_JSON','Expecting ','Unterminated string','Extra data')
    return bool(errors) and all(str(e).startswith(prefixes) for e in errors)

def mechanical_repair_messages(c,invalid_content,errors):
    research=[e for e in EVIDENCE if e['id'] in RESEARCH_IDS]
    obj={
      'case':c,'simulation_policy':POLICY,'OUTPUT_SCHEMA':OUTPUT_SCHEMA,
      'evidence_table':research,'optional_action_bases':[e for e in EVIDENCE if e['id'] in ['E04','E05']],
      'calculation_conventions':[e for e in EVIDENCE if e['id']=='E06'],
      'environment_measurement_conventions':[ENVIRONMENT_MEASUREMENT_CONVENTION],
      'original_invalid_output':invalid_content,'validation_errors':errors,
      'repair_instruction':'Mechanically repair JSON/schema/comparison arithmetic only. Preserve environmental recommendations, clinical-review flag, evidence choices, certainty and narrative meaning unless the listed mechanical error directly prevents valid representation. Return one complete JSON object with no duplicate keys.'
    }
    sys=COMMON+'\nThis is a final mechanical structured-output repair. No tools are available. Do not add new environmental or clinical reasoning merely to make the output valid.'
    return [{'role':'system','content':sys},{'role':'user','content':encode(obj)}]

def run_workflow(c,arm,client,call_limit=None):
    require(arm=='agent','Experiment 2 final notebook runs agent only')
    started=time.monotonic();messages=messages_for(c,'agent')
    result={'case_id':c['case_id'],'arm':'agent','model':MODEL,'started':stamp(),
      'status':'RUNNING','first_valid':None,'final_valid':False,
      'compact_correction_used':False,'mechanical_repair_used':False,
      'attempts':[],'tools':[],'final_attempts':[],'accepted':None,'settings':None}
    maxcalls=min(call_limit,AGENT_MAX_CALLS) if call_limit is not None else AGENT_MAX_CALLS
    ntools=0;seen_ids=set();available_research=set(RESEARCH_IDS)
    initial_final_content=None; initial_errors=None
    # Tool-capable initial workflow only.
    for turn in range(maxcalls):
        response=client.call(messages,True,f"{c['case_id']}/agent/initial/{turn+1}")
        result['attempts'].extend(response.get('attempts',[]))
        if response.get('error'):
            result['status']=response['error'];break
        choice=response['choice'];message=choice.get('message') or {}
        if choice.get('finish_reason')=='length':result['status']='OUTPUT_TRUNCATED';break
        if choice.get('finish_reason') in ['error','content_filter']:
            result['status']='GENERATION_'+choice['finish_reason'].upper();break
        if not isinstance(message,dict):result['status']='MALFORMED_RESPONSE';break
        calls=message.get('tool_calls') or []
        if not isinstance(calls,list):result['status']='INVALID_TOOL_CALL';break
        if calls:
            if ntools+len(calls)>AGENT_MAX_TOOLS:result['status']='TOOL_LIMIT';break
            assistant={k:copy.deepcopy(message[k]) for k in ['role','content','tool_calls','reasoning_details','reasoning'] if k in message}
            assistant['role']='assistant';messages.append(assistant)
            failed=False
            for tc in calls:
                name=None
                try:
                    require(isinstance(tc,dict) and tc.get('type')=='function' and isinstance(tc.get('function'),dict),'Malformed native tool call')
                    name=tc['function'].get('name');rawargs=tc['function'].get('arguments')
                    if not tc.get('id') or tc['id'] in seen_ids:raise ValueError('Missing/duplicate tool ID')
                    seen_ids.add(tc['id'])
                    if name not in TOOL_SCHEMAS:raise ValueError('Undeclared tool')
                    args=parse_final(rawargs) if isinstance(rawargs,str) else rawargs
                    Draft202012Validator(TOOL_SCHEMAS[name]).validate(args)
                    output=tool(name,args,c)
                    if name=='retrieve_evidence':available_research.update(e['id'] for e in output if e['id'] in RESEARCH_IDS)
                    result['tools'].append({'name':name,'args':args,'output':output,'success':True});ntools+=1
                    messages.append({'role':'tool','tool_call_id':tc['id'],'content':encode(output)})
                except Exception as exc:
                    result['tools'].append({'name':name,'success':False,'error':str(exc)})
                    result['status']='INVALID_TOOL_CALL';failed=True;break
            if failed:break
            continue
        initial_final_content=message.get('content')
        answer,assessment,errors,settings=evaluate_content(initial_final_content,c,available_research)
        valid=not errors;result['first_valid']=valid
        result['final_attempts'].append({'stage':'initial','answer':answer,'content':initial_final_content,
          'errors':errors,'valid':valid,'assessment':assessment,'available_research_ids':sorted(available_research)})
        if valid:
            result.update(status='VALID_SIMULATION',final_valid=True,accepted=answer,settings=settings);break
        initial_errors=errors
        # Do not accumulate the entire conversation for correction. Exit initial loop and use compact payload.
        break
    else:
        result['status']='CALL_LIMIT'

    # Compact correction for any invalid initial final answer.
    if result['first_valid'] is False and not result['final_valid'] and initial_final_content is not None:
        result['compact_correction_used']=True
        response=client.call(compact_repair_messages(c,initial_final_content,initial_errors),False,
                             f"{c['case_id']}/agent/compact_correction")
        result['attempts'].extend(response.get('attempts',[]))
        if not response.get('error'):
            choice=response['choice'];message=choice.get('message') or {};content=message.get('content')
            answer,assessment,errors,settings=evaluate_content(content,c,available_research)
            valid=not errors
            result['final_attempts'].append({'stage':'compact_correction','answer':answer,'content':content,
              'errors':errors,'valid':valid,'assessment':assessment,'available_research_ids':sorted(available_research)})
            if valid:
                result.update(status='VALID_SIMULATION',final_valid=True,accepted=answer,settings=settings)
            elif mechanical_errors_only(errors):
                # One bounded purely mechanical repair, tested separately in the parallel technical follow-up.
                result['mechanical_repair_used']=True
                response2=client.call(mechanical_repair_messages(c,content,errors),False,
                                      f"{c['case_id']}/agent/mechanical_repair")
                result['attempts'].extend(response2.get('attempts',[]))
                if not response2.get('error'):
                    choice2=response2['choice'];message2=choice2.get('message') or {};content2=message2.get('content')
                    answer2,assessment2,errors2,settings2=evaluate_content(content2,c,available_research)
                    valid2=not errors2
                    result['final_attempts'].append({'stage':'mechanical_repair','answer':answer2,'content':content2,
                      'errors':errors2,'valid':valid2,'assessment':assessment2,'available_research_ids':sorted(available_research)})
                    if valid2:result.update(status='VALID_SIMULATION',final_valid=True,accepted=answer2,settings=settings2)
                    else:result['status']='STAFF_REVIEW_REQUIRED'
                else:result['status']=response2['error']
            else:
                result['status']='STAFF_REVIEW_REQUIRED'
        else:
            result['status']=response['error']

    if result['status']=='RUNNING':result['status']='STAFF_REVIEW_REQUIRED'
    result['available_research_ids']=sorted(available_research)
    result['elapsed_seconds']=time.monotonic()-started
    result['http_requests']=len(result['attempts'])
    result['providers']=sorted({a['raw']['provider'] for a in result['attempts'] if isinstance(a.get('raw'),dict) and a['raw'].get('provider')})
    result['usage_records']=[a['raw']['usage'] for a in result['attempts'] if isinstance(a.get('raw'),dict) and isinstance(a['raw'].get('usage'),dict)]
    costs=[u.get('cost') for u in result['usage_records'] if type(u.get('cost')) in [int,float] and math.isfinite(u['cost']) and u['cost']>=0]
    result['known_cost_usd']=sum(costs)
    result['all_attempt_costs_known']=len(costs)==len(result['attempts']) and bool(costs)
    return result

def execute_batch(client):
    results=[]
    paths=[(c,repeat,OUT/f"{c['case_id']}_agent_r{repeat}.json") for repeat in range(1,REPEATS+1) for c in SELECTED]
    cached={}
    for c,repeat,path in paths:
        if path.exists():
            r=json.loads(path.read_text())
            require(r.get('case_hash')==digest(c) and r.get('fingerprint')==FINGERPRINT,'Cached case/configuration differs; use a new RUN_LABEL.')
            require(r.get('status')!='RUNNING','Interrupted workflow: inspect requests before any new execution.')
            cached[str(path)]=r
    halt=any(r.get('status') in HALT_STATUSES for r in cached.values())
    for c,repeat,path in paths:
        if str(path) in cached:results.append(cached[str(path)]);continue
        if halt:continue
        require(client is not None,'Missing client')
        save_json(path,{'status':'RUNNING','case_hash':digest(c),'fingerprint':FINGERPRINT,'started':stamp()})
        print('Running:',c['case_id'],'agent','r'+str(repeat),flush=True)
        r=run_workflow(c,'agent',client);r.update(case_hash=digest(c),repeat=repeat,fingerprint=FINGERPRINT)
        save_json(path,r);results.append(r)
        print(' →',r['status'],'| first valid:',r['first_valid'],'| compact:',r['compact_correction_used'],
              '| mechanical:',r['mechanical_repair_used'],'| requests:',r['http_requests'],'| cost $',round(r['known_cost_usd'],6),flush=True)
        if r['status'] in HALT_STATUSES:halt=True
    return results


In [ ]:
# 11 · Mandatory offline tests — zero network/model calls
OFFLINE_CHECKS_PASSED=False
require(RUN_LABEL=='AHR_Experiment2_FinalAgent_20261026_v4','Use a new run label for any design change')
require(RESULTS_ZIP_NAME=='AHR_EXPERIMENT2_FINAL_AGENT_V4_RESULTS.zip','Separate Experiment 2 archive required')
require(len(CASES)==12 and REPEATS==5,'Experiment 2 requires 12 cases × 5 repeats')
require(sum(e['kind']=='research' for e in EVIDENCE)==18,'Expected 18 curated research entries')
require(ENVIRONMENT_MEASUREMENT_CONVENTION['id']=='E36','E36 measurement convention missing')
# Explicitly prove no nurse-export field/name is present in the model prompt payload.
for c in CASES:
    prompt_blob=encode(messages_for(c,'agent'))
    for forbidden in ['reviewer_code','clinical_assessment_needed','receipt_id','years_in_nursing','clinical_setting_ward','NURSE-001','nurse_recommendation']:
        require(forbidden not in prompt_blob,'Nurse comparator leaked into model prompt: '+forbidden)

# Reuse state-aware retained-sound behavior on Experiment 2 cases.
def _base_answer(c):
    cmp=comparisons(c);night=int(c['context']['local_time'][:2])>=POLICY['night_start'] or int(c['context']['local_time'][:2])<POLICY['night_end']
    la='DECREASE' if night and c['room']['brightness']>c['staff']['brightness_min'] else ('INCREASE' if (not night and c['context']['goal']!='sleep' and c['room']['brightness']<c['staff']['brightness_max']) else 'NO_CHANGE')
    leid=['E01'] if la!='NO_CHANGE' else []
    lpid=[] if la!='NO_CHANGE' else ['E04']
    return {'physiological_interpretation':'Descriptive paired five-minute changes only.',
      'physiology_evidence_ids':['E03','E13'],'certainty':'MEDIUM','certainty_rationale':'A contextual choice exists but individual benefit remains uncertain.',
      'activity_assessment':c['context']['reported_activity'],
      'clinical_review':{'recommended':'NO','rationale':'No diagnostic conclusion is made from these synthetic comparisons.','evidence_ids':['E03','E13']},
      'comparisons':cmp,
      'lighting':{'action':la,'tone':c['preferences']['preferred_tones'][0] if c['preferences']['preferred_tones'] else c['room']['tone'],
                  'rationale':'Contextual simulated lighting choice.','evidence_ids':leid,'policy_ids':lpid},
      'sound':{'option':'NO_CHANGE','rationale':'Retain current sound state.','evidence_ids':[],'policy_ids':['E05']},
      'limitations':['Synthetic inputs and research summaries do not establish individual clinical benefit.']}

# Active retained sound requires matching research.
c=next(x for x in CASES if x['case_id']=='E2_001');a=_base_answer(c);a['sound']={'option':'NO_CHANGE','rationale':'Retain current nature sounds.','evidence_ids':['E02'],'policy_ids':[]}
err,_=validate_answer(a,c,RESEARCH_IDS);require(not err,'Retained nature + E02 should validate: '+str(err))
a=_base_answer(c);a['sound']={'option':'NO_CHANGE','rationale':'Retain current nature sounds.','evidence_ids':['E07'],'policy_ids':[]}
err,_=validate_answer(a,c,RESEARCH_IDS);require(any(e.startswith('EVIDENCE_ACTION_MISMATCH sound/E07') for e in err),'Wrong retained-sound evidence must fail')
# Retained music.
c=next(x for x in CASES if x['case_id']=='E2_005');a=_base_answer(c);a['sound']={'option':'NO_CHANGE','rationale':'Retain current preferred calming music.','evidence_ids':['E07'],'policy_ids':['E05']}
err,_=validate_answer(a,c,RESEARCH_IDS);require(not err,'Retained music + E07 should validate: '+str(err))
# Retained silence under opt-out uses prototype basis only.
c=next(x for x in CASES if x['case_id']=='E2_009');a=_base_answer(c);a['sound']={'option':'NO_CHANGE','rationale':'Retain silence under recorded audio opt-out.','evidence_ids':[],'policy_ids':['E05']}
err,_=validate_answer(a,c,RESEARCH_IDS);require(not err,'Retained silence + E05 should validate: '+str(err))
# Clinical review has no embedded expected answer: both YES and NO are schema-valid when other fields are unchanged.
for flag in ['YES','NO']:
    aa=copy.deepcopy(a);aa['clinical_review']['recommended']=flag
    errs,_=validate_answer(aa,c,RESEARCH_IDS);require(not errs,'Clinical-review schema should not encode a gold-standard flag: '+str(errs))

# Preflight all 12 initial prompts plus conservative compact/mechanical repair probes under unchanged 100k guard.
probe=Router('offline-fake',MASTER_ROOT/'_offline_preflight',[{'tag':'dummy','provider_name':PROVIDER}])
PREFLIGHT=[]
for c in CASES:
    initial=messages_for(c,'agent'); est,_=probe.reservation(probe.payload(initial,True));require(est<MAX_ESTIMATED_INPUT_TOKENS,'Initial prompt exceeds guard '+c['case_id'])
    dummy='{"invalid":"synthetic preflight placeholder"}';errs=['SCHEMA []: synthetic preflight']
    compact=compact_repair_messages(c,dummy,errs);est2,_=probe.reservation(probe.payload(compact,False));require(est2<MAX_ESTIMATED_INPUT_TOKENS,'Compact repair exceeds guard '+c['case_id'])
    mech=mechanical_repair_messages(c,dummy,['DUPLICATE_JSON_KEY percent_change']);est3,_=probe.reservation(probe.payload(mech,False));require(est3<MAX_ESTIMATED_INPUT_TOKENS,'Mechanical repair exceeds guard '+c['case_id'])
    PREFLIGHT.append({'case_id':c['case_id'],'initial_guard_units':est,'compact_guard_units':est2,'mechanical_guard_units':est3})
pd.DataFrame(PREFLIGHT).to_csv(OUT/'offline_preflight.csv',index=False)
OFFLINE_CHECKS_PASSED=True
print('OFFLINE TESTS PASS | 12 cases | 60 planned | max prompt guard units:',max(max(x['initial_guard_units'],x['compact_guard_units'],x['mechanical_guard_units']) for x in PREFLIGHT),'| paid calls: 0')
display(pd.DataFrame(PREFLIGHT))


In [ ]:
# 12 · Frozen run contract, smoke gate and resume-safe execution
RUN_IDENTITY={
 'version':'AHR-Experiment2-final-agent-v4.0',
 'case_snapshot_sha256':'da7a13dc7d6cac3c80979cb61f8d33eb5fb307e388ecb25b70e586612a054105',
 'case_source_workbook_sha256':'9a6aca612b5d6bb91ccf5bf9e849a34144803965da2ae0464dce30989af19ae3',
 'evidence_source_workbook_sha256':'b96fd65f74cbea6e17fe50eb26b4d8e591c816286ca0a7a1d6a8e6376ae75bdf',
 'case_ids':CASE_IDS,'repeats':REPEATS,'model':MODEL,'provider':PROVIDER,
 'settings':{k:globals()[k] for k in ['REASONING_EFFORT','SEND_TEMPERATURE','TEMPERATURE','MAX_OUTPUT_TOKENS','AGENT_MAX_CALLS','AGENT_MAX_TOOLS','TRANSIENT_RETRIES','REQUEST_TIMEOUT','PACE_SECONDS','RUN_BUDGET_USD','GLOBAL_BUDGET_USD','MAX_REQUEST_RESERVATION_USD','MAX_ESTIMATED_INPUT_TOKENS','MAX_HTTP_REQUESTS','MAX_INPUT_PRICE_PER_M','MAX_OUTPUT_PRICE_PER_M']},
 'design':'12 researcher-authored synthetic cases: 3 base situations × 2 environment conditions × 2 physiology conditions; one final agentic workflow × 5 repeats',
 'evidence':'18 curated research entries + E04-E06 prototype rules + separate E36 lux/dBA measurement convention',
 'technical_architecture':'state-aware retained-sound validation + compact correction + bounded mechanical repair',
 'clinical_review_field':'YES/NO advisory, non-diagnostic; no expected case flag encoded',
 'nurse_alignment':'POST_RUN_ONLY; nurse comparator not loaded by inference cells'
}
write_frozen(OUT/'run_identity.json',RUN_IDENTITY)
# Deterministic implementation fingerprint.
# Do not use repr(code.co_consts): nested code-object reprs can contain memory
# addresses and change across otherwise identical Colab/kernel executions.
import types

def _stable_code_payload(code):
    def norm(x):
        if isinstance(x, types.CodeType):
            return _stable_code_payload(x)
        if isinstance(x, bytes):
            return {'__bytes__': x.hex()}
        if isinstance(x, tuple):
            return [norm(v) for v in x]
        if isinstance(x, list):
            return [norm(v) for v in x]
        if isinstance(x, dict):
            return {str(k): norm(v) for k,v in sorted(x.items(), key=lambda kv: str(kv[0]))}
        if isinstance(x, (set, frozenset)):
            vals=[norm(v) for v in x]
            return sorted(vals, key=lambda v: encode(v))
        if x is None or isinstance(x, (str,int,float,bool)):
            return x
        return {'__type__': type(x).__name__, '__value__': str(x)}
    return {
        'co_code': code.co_code.hex(),
        'co_consts': [norm(v) for v in code.co_consts],
        'co_names': list(code.co_names),
        'co_varnames': list(code.co_varnames),
        'co_freevars': list(code.co_freevars),
        'co_cellvars': list(code.co_cellvars),
        'co_argcount': code.co_argcount,
        'co_posonlyargcount': getattr(code,'co_posonlyargcount',0),
        'co_kwonlyargcount': code.co_kwonlyargcount,
        'co_flags': code.co_flags,
    }

def stable_function_hash(f):
    return digest({'name':getattr(f,'__name__',''), 'code':_stable_code_payload(f.__code__)})

CONFIG={'version':'AHR-Experiment2-final-agent-contract-4.0','identity':RUN_IDENTITY,'evidence_table':EVIDENCE,
 'environment_measurement_convention':ENVIRONMENT_MEASUREMENT_CONVENTION,'policy':POLICY,
 'common_prompt':COMMON,'tools':TOOLS,'output_schema':OUTPUT_SCHEMA,
 'implementation':{n:stable_function_hash(f) for n,f in
    [('comparisons',comparisons),('tool',tool),('validate_answer',validate_answer),('assess_answer',assess_answer),
     ('messages_for',messages_for),('run_workflow',run_workflow),('parse_final',parse_final),
     ('compact_repair_messages',compact_repair_messages),('mechanical_repair_messages',mechanical_repair_messages),('eligible_endpoints',eligible_endpoints)]}}
FINGERPRINT=digest(CONFIG)
write_frozen(OUT/'experiment_contract.json',{'fingerprint':FINGERPRINT,'config':CONFIG})

def check_smoke():
    p=OUT/'E2_PILOT_001_agent_r1.json'
    require(p.exists(),'Run smoke stage before full')
    r=json.loads(p.read_text());require(r.get('fingerprint')==FINGERPRINT and r.get('case_hash')==digest(SMOKE_CASE),'Smoke version mismatch')
    require(r.get('status')=='VALID_SIMULATION' and r.get('final_valid') is True,'Smoke failed. Inspect before full run.')

SELECTED=([SMOKE_CASE] if RUN_STAGE=='smoke' else CASES)
_BATCH_REPEATS=REPEATS
if RUN_STAGE=='smoke':REPEATS=1
if RUN_STAGE=='full':check_smoke()
if RUN_STAGE!='offline':
    require(OFFLINE_CHECKS_PASSED and APPROVE_PAID_CALLS,'Offline checks and paid approval required')
    with generation_lock(OUT):
        client=prepare_client()
        write_frozen(OUT/'manifest.json',{'fingerprint':FINGERPRINT,'cases':CASES,'config':CONFIG,'development_smoke':SMOKE_CASE})
        BATCH_RESULTS=execute_batch(client)
    print('Stage',RUN_STAGE,'complete or stopped; saved workflows:',len(BATCH_RESULTS))
else:
    print('OFFLINE: validation/setup complete. No API key needed, no paid requests.')
REPEATS=_BATCH_REPEATS


In [ ]:
# 13 · Nurse-free model results and reliability metrics
RESULT_ROWS=[]
for c in CASES:
    for rep in range(1,REPEATS+1):
        path=OUT/f"{c['case_id']}_agent_r{rep}.json";r=json.loads(path.read_text()) if path.exists() else {};a=r.get('accepted') or {};cfg=r.get('settings') or {}
        RESULT_ROWS.append({'case_id':c['case_id'],'base_situation':c['design_labels']['base_situation'],
          'environment_condition':c['design_labels']['environment_condition'],'physiology_condition':c['design_labels']['physiology_condition'],
          'repeat':rep,'status':r.get('status','NOT_RUN'),'first_valid':r.get('first_valid'),'final_valid':r.get('final_valid'),
          'compact_correction_used':r.get('compact_correction_used'),'mechanical_repair_used':r.get('mechanical_repair_used'),
          'http_requests':r.get('http_requests'),'workflow_cost_known_usd':r.get('known_cost_usd'),'elapsed_seconds':r.get('elapsed_seconds'),
          'lighting_action':(a.get('lighting') or {}).get('action'),'light_tone':(a.get('lighting') or {}).get('tone'),
          'sound_option':(a.get('sound') or {}).get('option'),'resulting_sound_state':cfg.get('sound'),
          'clinical_review_recommended':(a.get('clinical_review') or {}).get('recommended'),
          'certainty':a.get('certainty')})
res=pd.DataFrame(RESULT_ROWS);res.to_csv(OUT/'run_results_60.csv',index=False)
summary=pd.DataFrame([{
  'planned':60,'attempted':int(sum(res.status!='NOT_RUN')),'first_pass_valid':int(sum(res.first_valid==True)),
  'final_valid':int(sum(res.final_valid==True)),'compact_correction_used':int(sum(res.compact_correction_used==True)),
  'mechanical_repair_used':int(sum(res.mechanical_repair_used==True)),
  'known_workflow_usd':float(res.workflow_cost_known_usd.fillna(0).sum()),
  'average_seconds':float(res.elapsed_seconds.mean()) if res.elapsed_seconds.notna().any() else None
}]);summary.to_csv(OUT/'run_summary.csv',index=False)
ledger_file=OUT/'cost_ledger.json';ledger=json.loads(ledger_file.read_text()) if ledger_file.exists() else []
run_note={'stage':RUN_STAGE,'case_count':12,'planned_runs':60,'attempted':int(sum(res.status!='NOT_RUN')),
          'first_pass_valid':int(sum(res.first_valid==True)),'final_valid':int(sum(res.final_valid==True)),
          'nurse_reference':'NOT_LOADED_IN_THIS_CELL','known_cost_usd':float(sum(x['accounted_usd'] for x in ledger if x.get('cost_known') is True))}
save_json(OUT/'progress_summary.json',run_note)
print(run_note);display(summary);display(res.head(12))
archive=BASE/RESULTS_ZIP_NAME
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in OUT.rglob('*'):
        if f.is_file() and f.name!='generation.lock':z.write(f,arcname=str(f.relative_to(OUT)))
print('MODEL RESULTS ZIP:',archive,'| MB',round(archive.stat().st_size/1e6,3))


In [ ]:
# 14 · POST-RUN ONLY: load frozen NURSE-001 comparator and calculate Experiment 2 alignment/sensitivity
# This cell is intentionally after model generation/results. Nothing here is used by messages_for(), tools, validation or repairs.
NURSE_CSV_DRIVE_PATH=Path('/content/drive/MyDrive/COMP6002 Adaptive Hospital Room/10_Experiment_2_Synthetic_Study/AHR_Experiment2_NURSE001_FROZEN_2026-10-26.csv')
EXPECTED_NURSE_SHA256='fd11d3cf2523cf43dff938db6e459d72b08277d040162bfbfa48469ee9c56cd5'

def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()

def mode_or_tie(values):
    s=pd.Series([v for v in values if pd.notna(v)])
    if s.empty:return None
    vc=s.value_counts();return vc.index[0] if len(vc)==1 or vc.iloc[0]>vc.iloc[1] else 'TIE'

def resulting_nurse_sound(row,current_sound):
    return current_sound if row['sound_recommendation']=='NO_CHANGE' else row['sound_recommendation']

if RUN_STAGE=='full' and int(sum(res.status!='NOT_RUN'))==60:
    require(NURSE_CSV_DRIVE_PATH.exists(),'Frozen nurse comparator not found at expected Drive path')
    require(file_sha256(NURSE_CSV_DRIVE_PATH)==EXPECTED_NURSE_SHA256,'Frozen nurse CSV hash mismatch')
    nurse=pd.read_csv(NURSE_CSV_DRIVE_PATH)
    require(len(nurse)==12 and nurse.case_id.nunique()==12 and set(nurse.case_id)==set(CASE_IDS),'Nurse comparator must contain exactly E2_001-E2_012')
    require(nurse.reviewer_code.nunique()==1,'Expected one frozen nurse reviewer')
    case_by_id={c['case_id']:c for c in CASES}
    nurse=nurse.copy()
    nurse['nurse_resulting_sound_state']=[resulting_nurse_sound(row,case_by_id[row.case_id]['room']['sound']) for _,row in nurse.iterrows()]
    nurse['nurse_clinical_review']=nurse['clinical_assessment_needed'].astype(str).str.upper()
    ncols=['case_id','brightness_recommendation','tone_recommendation','sound_recommendation','nurse_resulting_sound_state','nurse_clinical_review','confidence']
    merged=res.merge(nurse[ncols],on='case_id',how='left',validate='many_to_one')
    valid=merged[merged.final_valid==True].copy()
    valid['agree_brightness']=valid.lighting_action==valid.brightness_recommendation
    valid['agree_tone']=valid.light_tone==valid.tone_recommendation
    valid['agree_sound_command']=valid.sound_option==valid.sound_recommendation
    valid['agree_resulting_sound']=valid.resulting_sound_state==valid.nurse_resulting_sound_state
    valid['agree_clinical_review']=valid.clinical_review_recommended==valid.nurse_clinical_review
    valid['agree_exact_command_triplet']=valid.agree_brightness & valid.agree_tone & valid.agree_sound_command
    valid['agree_exact_resulting_triplet']=valid.agree_brightness & valid.agree_tone & valid.agree_resulting_sound
    valid.to_csv(OUT/'nurse_alignment_run_level.csv',index=False)

    alignment={'valid_model_runs':int(len(valid)),'planned_model_runs':60}
    for col in ['agree_brightness','agree_tone','agree_sound_command','agree_resulting_sound','agree_clinical_review','agree_exact_command_triplet','agree_exact_resulting_triplet']:
        alignment[col]={'n':int(valid[col].sum()),'denominator':int(len(valid)),'percent':float(100*valid[col].mean()) if len(valid) else None}

    # Case-level modal recommendations across five repeats.
    modes=[]
    for cid,g in valid.groupby('case_id'):
        nr=nurse[nurse.case_id==cid].iloc[0]
        row={'case_id':cid,'valid_repeats':len(g)}
        for src,dst in [('lighting_action','model_brightness_mode'),('light_tone','model_tone_mode'),('sound_option','model_sound_command_mode'),('resulting_sound_state','model_resulting_sound_mode'),('clinical_review_recommended','model_clinical_review_mode')]:row[dst]=mode_or_tie(g[src])
        row.update({'nurse_brightness':nr.brightness_recommendation,'nurse_tone':nr.tone_recommendation,'nurse_sound_command':nr.sound_recommendation,
                    'nurse_resulting_sound':nr.nurse_resulting_sound_state,'nurse_clinical_review':nr.nurse_clinical_review})
        row['case_mode_exact_resulting_triplet']=(row['model_brightness_mode']==row['nurse_brightness'] and row['model_tone_mode']==row['nurse_tone'] and row['model_resulting_sound_mode']==row['nurse_resulting_sound'])
        row['case_mode_clinical_review_agreement']=row['model_clinical_review_mode']==row['nurse_clinical_review']
        modes.append(row)
    modes=pd.DataFrame(modes).sort_values('case_id');modes.to_csv(OUT/'nurse_alignment_case_modes.csv',index=False)

    # Repeat stability.
    stab=[]
    for cid,g in valid.groupby('case_id'):
        row={'case_id':cid,'valid_repeats':len(g)}
        for field in ['lighting_action','light_tone','sound_option','resulting_sound_state','clinical_review_recommended']:
            m=mode_or_tie(g[field]);row[field+'_mode']=m;row[field+'_mode_share']=float((g[field]==m).mean()) if m!='TIE' else None
        stab.append(row)
    stab=pd.DataFrame(stab).sort_values('case_id');stab.to_csv(OUT/'repeat_stability_by_case.csv',index=False)

    # Controlled pair sensitivity. Compare whether the model changes a choice when only one factor changes; match that change/no-change pattern to nurse.
    env_pairs=[('E2_001','E2_003'),('E2_002','E2_004'),('E2_005','E2_007'),('E2_006','E2_008'),('E2_009','E2_011'),('E2_010','E2_012')]
    phys_pairs=[('E2_001','E2_002'),('E2_003','E2_004'),('E2_005','E2_006'),('E2_007','E2_008'),('E2_009','E2_010'),('E2_011','E2_012')]
    fields=[('lighting_action','brightness_recommendation'),('light_tone','tone_recommendation'),('resulting_sound_state','nurse_resulting_sound_state'),('clinical_review_recommended','nurse_clinical_review')]
    pair_rows=[]
    for factor,pairs in [('environment',env_pairs),('physiology',phys_pairs)]:
        for a_id,b_id in pairs:
            na=nurse[nurse.case_id==a_id].iloc[0];nbb=nurse[nurse.case_id==b_id].iloc[0]
            for rep in range(1,REPEATS+1):
                ga=valid[(valid.case_id==a_id)&(valid.repeat==rep)];gb=valid[(valid.case_id==b_id)&(valid.repeat==rep)]
                if len(ga)!=1 or len(gb)!=1:continue
                ga=ga.iloc[0];gb=gb.iloc[0]
                for mf,nf in fields:
                    model_changed=(ga[mf]!=gb[mf]);nurse_changed=(na[nf]!=nbb[nf])
                    pair_rows.append({'factor':factor,'case_a':a_id,'case_b':b_id,'repeat':rep,'field':mf,
                                      'model_changed':bool(model_changed),'nurse_changed':bool(nurse_changed),'change_pattern_agreement':bool(model_changed==nurse_changed)})
    pair_df=pd.DataFrame(pair_rows);pair_df.to_csv(OUT/'controlled_pair_sensitivity.csv',index=False)
    pair_summary=pair_df.groupby(['factor','field']).agg(comparisons=('change_pattern_agreement','size'),agreement=('change_pattern_agreement','sum'),agreement_rate=('change_pattern_agreement','mean'),model_change_rate=('model_changed','mean'),nurse_change_rate=('nurse_changed','mean')).reset_index()
    pair_summary.to_csv(OUT/'controlled_pair_sensitivity_summary.csv',index=False)

    analysis={'nurse_csv_sha256':EXPECTED_NURSE_SHA256,'reviewer_count':int(nurse.reviewer_code.nunique()),'alignment':alignment,
              'case_mode_exact_resulting_triplet_n':int(modes.case_mode_exact_resulting_triplet.sum()),'case_mode_denominator':int(len(modes)),
              'case_mode_clinical_review_agreement_n':int(modes.case_mode_clinical_review_agreement.sum()),
              'important':'Nurse comparator was loaded only after all model records existed; it was not model-visible and was not used by the validator.'}
    save_json(OUT/'experiment2_nurse_comparison_summary.json',analysis)
    print('POST-RUN NURSE COMPARISON COMPLETE');print(json.dumps(analysis,indent=2));display(modes);display(pair_summary)
    # Rebuild archive including post-run analysis files.
    archive=BASE/RESULTS_ZIP_NAME
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for f in OUT.rglob('*'):
            if f.is_file() and f.name!='generation.lock':z.write(f,arcname=str(f.relative_to(OUT)))
    print('FINAL ZIP WITH NURSE ANALYSIS:',archive,'| MB',round(archive.stat().st_size/1e6,3))
    if DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(str(archive))
        except (ImportError,ModuleNotFoundError):
            print('Outside Colab. Archive remains at:',archive)
else:
    print('Nurse comparator NOT loaded. This cell runs only after a complete 60-run full stage.')


## How to run

1. Run all cells with `RUN_STAGE='offline'` and `APPROVE_PAID_CALLS=False`. Confirm all offline tests pass. The nurse CSV is not loaded.
2. Set `RUN_STAGE='smoke'` and `APPROVE_PAID_CALLS=True`; run all cells. One development-only agent workflow (`E2_PILOT_001`) must finish valid.
3. Set `RUN_STAGE='full'` and keep `APPROVE_PAID_CALLS=True`; run all cells. The formal stage executes **12 cases × 5 repeats = 60 agent workflows** and is resume-safe on Drive.
4. Cell 14 loads the frozen NURSE-001 CSV **only after all 60 formal records exist**, verifies its SHA-256, and writes run-level alignment, case-mode alignment, repeat stability, controlled environment/physiology sensitivity, and clinical-review agreement.
5. Do not change cases, evidence, prompt, schema, validator, repair rules or model settings after paid smoke begins. Any substantive change requires a new run label/version.
